In [ ]:
#FNIRS analysis 

In [6]:
#1. Preparing the dataframes 

import pandas as pd
from scipy.stats import ttest_1samp, spearmanr, ttest_rel
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import os 

BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
BEHAVIOURZSCORE.rename(columns={'Participant': 'Participant Number'}, inplace=True)
data = pd.read_excel('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Debriefsummaryscores/summary_scores.xlsx')
data.rename(columns={'Participant': 'Participant Number'}, inplace=True)
BETAS = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/BETAVALUES2.csv")
REGRESSOR1HUNT = BETAS[BETAS['Index'] == 'Sn(1) ChasexDirectionToPreyHunt^1*bf(1)']
REGRESSOR1FOLLOW = BETAS[BETAS['Index'] == 'Sn(1) FollowxDirectionToPreyFollow^1*bf(1)']
REGRESSOR2HUNT = BETAS[BETAS['Index'] == 'Sn(1) ChasexDirectionToOtherHunt^1*bf(1)']
REGRESSOR2FOLLOW = BETAS[BETAS['Index'] == 'Sn(1) FollowxDirectionToOtherFollow^1*bf(1)']
REGRESSOR3HUNT = BETAS[BETAS['Index'] == 'Sn(1) ChasexPreyDistHunt^1*bf(1)']
REGRESSOR3FOLLOW = BETAS[BETAS['Index'] == 'Sn(1) FollowxPreyDistFollow^1*bf(1)']
REGRESSOR4HUNT = BETAS[BETAS['Index'] == 'Sn(1) ChasexOtherDistHunt^1*bf(1)']
REGRESSOR4FOLLOW = BETAS[BETAS['Index'] == 'Sn(1) FollowxOtherDistFollow^1*bf(1)']

suffix = "Matrix6"

def merge_and_rename(df, behaviour_data, data, keep_columns):
    behaviour_data = behaviour_data[['Dyad', 'Participant Number', 'Hunting Success']]
    merged_data = pd.merge(data, behaviour_data, on='Participant Number', how='left')
    merged = df.merge(merged_data, on=['Dyad', 'Participant Number'], how='left')
    merged.fillna(0, inplace=True)
    renamed = merged.rename(columns={col: 'Channel ' + col if col not in keep_columns else col 
                                     for col in merged.columns})
    return renamed, merged

keep_columns = ['Dyad', 'Hunting Success', 'Participant Number', 'Index'] + list(data.columns)

behaviour_grouped = (
    BEHAVIOURZSCORE
    .drop(columns=['Event']) 
    .groupby(['Dyad', 'Participant Number'])
    .mean()
    .reset_index()
)

REGRESSOR1HUNT, REGRESSOR1HUNTRAW = merge_and_rename(REGRESSOR1HUNT, behaviour_grouped, data, keep_columns)
REGRESSOR1FOLLOW, REGRESSOR1FOLLOWRAW = merge_and_rename(REGRESSOR1FOLLOW, behaviour_grouped, data, keep_columns)
REGRESSOR2HUNT, REGRESSOR2HUNTRAW = merge_and_rename(REGRESSOR2HUNT, behaviour_grouped, data, keep_columns)
REGRESSOR2FOLLOW, REGRESSOR2FOLLOWRAW = merge_and_rename(REGRESSOR2FOLLOW, behaviour_grouped, data, keep_columns)
REGRESSOR3HUNT, REGRESSOR3HUNTRAW = merge_and_rename(REGRESSOR3HUNT, behaviour_grouped, data, keep_columns)
REGRESSOR3FOLLOW, REGRESSOR3FOLLOWRAW = merge_and_rename(REGRESSOR3FOLLOW, behaviour_grouped, data, keep_columns)
REGRESSOR4HUNT, REGRESSOR4HUNTRAW = merge_and_rename(REGRESSOR4HUNT, behaviour_grouped, data, keep_columns)
REGRESSOR4FOLLOW, REGRESSOR4FOLLOWRAW = merge_and_rename(REGRESSOR4FOLLOW, behaviour_grouped, data, keep_columns)

exclude_columns = ['Dyad', 'Participant Number', 'Index', 'Hunting Success','Planning', 'Surprise','Expectation confirmation',
    'Theory of Mind', 'Happy',
    'Frustrated']
exclude_calculation = ['Dyad', 'Participant Number', 'Index','Planning','Surprise','Expectation confirmation','Theory of Mind','Happy','Frustrated']

def perform_ttest_against_success_and_save(data, filename):
    columns, correlations, p_corr_values = [], [], []
    success_column = 'Hunting Success'
    if success_column not in data.columns:
        raise ValueError(f"The column '{success_column}' is missing from the dataset.")
    for column in data.columns:
        if column not in exclude_columns and column != success_column:
            columns.append(column) 
            try:
                corr, p_corr = spearmanr(data[column], data[success_column])
                correlations.append(round(corr, 3))
                p_corr_values.append(round(p_corr, 3))
            except ValueError as e:
                print(f"Error in '{data}' with column '{column}': {e}")
                correlations.append(None)
                p_corr_values.append(None)
    _, p_corr_corrected, _, _ = multipletests(p_corr_values, method='fdr_bh') if p_corr_values else ([], [], [], [])
    results_df = pd.DataFrame({
        'Channel': columns,
        'Correlation with Hunting Success': correlations,
        'p_corr': p_corr_values,
        'p_corr_corrected': p_corr_corrected
    })
    results_df.to_csv(filename, index=False)

def perform_ttest_against_zero_and_save(data, filename):
    columns, t_statistics, p_values = [], [], []
    for column in data.columns:
        if column not in exclude_columns:
            t_stat, p_val = ttest_1samp(data[column], 0)
            t_statistics.append(round(t_stat, 2))
            p_values.append(round(p_val, 3))
            columns.append(column)
    _, p_vals_corrected, _, _ = multipletests(p_values, method='fdr_bh')
    results_df = pd.DataFrame({
        'Channel': columns,
        't': t_statistics,
        'p': p_values,
        'p_corrected': p_vals_corrected
    })
    results_df.to_csv(filename, index=False)

def perform_ttest_against_zero_and_save_notcorrected(data, filename):
    columns, t_statistics, p_values = [], [], []
    for column in data.columns:
        if column not in exclude_columns:
            t_stat, p_val = ttest_1samp(data[column], 0)
            t_statistics.append(round(t_stat, 2))
            p_values.append(round(p_val, 3))
            columns.append(column)
    p_vals_corrected = p_values
    results_df = pd.DataFrame({
        'Channel': columns,
        't': t_statistics,
        'p': p_values,
        'p_corrected': p_vals_corrected
    })
    results_df.to_csv(filename, index=False)

output_base_path = '/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/'

frame_pairs = [
    (REGRESSOR1HUNT, REGRESSOR1FOLLOW, f"{output_base_path}REGRESSOR1_HUNT_FOLLOW_{suffix}.csv", True),
    (REGRESSOR2HUNT, REGRESSOR2FOLLOW, f"{output_base_path}REGRESSOR2_HUNT_FOLLOW_{suffix}.csv", True),
    (REGRESSOR3HUNT, REGRESSOR3FOLLOW, f"{output_base_path}REGRESSOR3_HUNT_FOLLOW_{suffix}.csv", True),
    (REGRESSOR4HUNT, REGRESSOR4FOLLOW, f"{output_base_path}REGRESSOR4_HUNT_FOLLOW_{suffix}.csv", True)
]

frames = [
    (REGRESSOR1HUNT, f"{output_base_path}REGRESSOR1_HUNT_{suffix}.csv", True),
    (REGRESSOR1FOLLOW, f"{output_base_path}REGRESSOR1_FOLLOW_{suffix}.csv", True),
    (REGRESSOR2HUNT, f"{output_base_path}REGRESSOR2_HUNT_{suffix}.csv", True),
    (REGRESSOR2FOLLOW, f"{output_base_path}REGRESSOR2_FOLLOW_{suffix}.csv", True),
    (REGRESSOR3HUNT, f"{output_base_path}REGRESSOR3_HUNT_{suffix}.csv", True),
    (REGRESSOR3FOLLOW, f"{output_base_path}REGRESSOR3_FOLLOW_{suffix}.csv", True),
    (REGRESSOR4HUNT, f"{output_base_path}REGRESSOR4_HUNT_{suffix}.csv", True),
    (REGRESSOR4FOLLOW, f"{output_base_path}REGRESSOR4_FOLLOW_{suffix}.csv", True)
]

for hunt_frame, follow_frame, output_path, calculate_corr in frame_pairs:
    hunt_frame_cleaned = hunt_frame.drop(columns=['Hunting Success']).drop(columns=exclude_calculation, errors='ignore')
    follow_frame_cleaned = follow_frame.drop(columns=['Hunting Success']).drop(columns=exclude_calculation, errors='ignore')
    hunt_frame_cleaned_reset = hunt_frame_cleaned.reset_index(drop=True)
    follow_frame_cleaned_reset = follow_frame_cleaned.reset_index(drop=True)
    ttest_results = {}
    p_values = []
    for column in hunt_frame_cleaned_reset.columns:
        if column in follow_frame_cleaned_reset.columns:
            t_stat, p_value = ttest_rel(hunt_frame_cleaned_reset[column], follow_frame_cleaned_reset[column])
            ttest_results[column] = {'t_stat': t_stat, 'p_value': p_value}
            p_values.append(p_value)
    _, p_values_corrected, _, _ = multipletests(p_values, method='fdr_bh')
    if 'REGRESSOR1' in output_path:
        p_values2 = p_values_corrected
    for i, (column, stats) in enumerate(ttest_results.items()):
        stats['p_value_corrected'] = p_values_corrected[i]
    ttest_results_df = pd.DataFrame.from_dict(ttest_results, orient='index').reset_index()
    ttest_results_df.columns = ['Feature', 't_stat', 'p_value', 'p_value_corrected']
    regressor_name = output_path.split('/')[-1].split('_')[0]  
    paired_csv_filename = f"{output_base_path}{regressor_name}_ttest_results.csv"
    ttest_results_df.to_csv(paired_csv_filename, index=False)

for frame, output_path, calculate_corr in frames:
    output_path_zero = output_path.replace('.csv', '_zero.csv')
    output_path_zero1 = output_path.replace('.csv', '_zeronotcorrected.csv')
    perform_ttest_against_zero_and_save(frame, output_path_zero)
    perform_ttest_against_zero_and_save_notcorrected(frame, output_path_zero1)
    if calculate_corr:
        output_path_success = output_path.replace('.csv', '_success.csv')
        perform_ttest_against_success_and_save(frame, output_path_success)

metadata_columns = ['Dyad', 'Participant Number', 'Hunting Success']
for frame_pair in frame_pairs:
    hunt_frame, follow_frame, output_path, calculate_corr = frame_pair
    hunt_metadata = hunt_frame[metadata_columns]
    follow_metadata = follow_frame[metadata_columns]
    if not hunt_metadata.equals(follow_metadata):
        raise ValueError(f"Metadata columns are not aligned between frames for {output_path}")
    hunt_numerical = hunt_frame.drop(columns=metadata_columns + ['Index'])
    follow_numerical = follow_frame.drop(columns=metadata_columns + ['Index'])
    if "HUNT_FOLLOW" in output_path:
        difference_numerical = hunt_numerical - follow_numerical
    elif "FOLLOW_HUNT" in output_path:
        difference_numerical = follow_numerical - hunt_numerical
    else:
        raise ValueError("Unexpected filename pattern; cannot determine subtraction order.")
    difference_frame = pd.concat([hunt_metadata, difference_numerical], axis=1)
    if calculate_corr:
        output_path_success = output_path.replace('.csv', '_success.csv')
        perform_ttest_against_success_and_save(difference_frame, output_path_success)

def perform_ttest_against_success_and_save2(data, filename, success_column, exclude_columns):
    columns, correlations, p_corr_values = [], [], []
    if success_column not in data.columns:
        raise ValueError(f"The column '{success_column}' is missing from the dataset.")
    for column in data.columns:
        if column not in exclude_columns and column != success_column:
            columns.append(column)
            try:
                corr, p_corr = spearmanr(data[column], data[success_column])
                correlations.append(round(corr, 3))
                p_corr_values.append(round(p_corr, 3))
            except ValueError as e:
                print(f"Error with column '{column}': {e}")
                correlations.append(None)
                p_corr_values.append(None)
    _, p_corr_corrected, _, _ = multipletests(p_corr_values, method='fdr_bh') if p_corr_values else ([], [], [], [])
    results_df = pd.DataFrame({
        'Channel': columns,
        'Correlation with ' + success_column: correlations,
        'p_corr': p_corr_values,
        'p_corr_corrected': p_corr_corrected
    })
    results_df.to_csv(filename, index=False)

success_columns = [
    'Planning', 'Surprise','Expectation confirmation',
    'Theory of Mind', 'Happy',
    'Frustrated'
]

output_path2 = '/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/REGRESSOR1_HUNT_FOLLOW_Matrix1.csv'
for success_column in success_columns:
    output_path_success = output_path2.replace('.csv', f'_{success_column.replace(" ", "_")}_successHUNT.csv')
    perform_ttest_against_success_and_save2(REGRESSOR1HUNT, output_path_success, success_column, exclude_columns)
    output_path_success = output_path2.replace('.csv', f'_{success_column.replace(" ", "_")}_successFOLLOW.csv')
    perform_ttest_against_success_and_save2(REGRESSOR1FOLLOW, output_path_success, success_column, exclude_columns)

def perform_linear_mixed_model_by_channel_and_save(data, output_dir, predictors, random_effects, exclude_columns):
    results = []  
    predictors_formula = " + ".join(predictors)
    for channel in data.columns:
        if channel not in exclude_columns:
            try:
                formula = f"{channel} ~ {predictors_formula}"
                model = smf.mixedlm(
                    formula=formula, 
                    data=data, 
                    groups=data[random_effects[0]],  
                    re_formula="1"  
                )
                result = model.fit()
                results = []
                for var in predictors:
                    results.append({
                        "Channel (y-variable)": channel,
                        "Predictor (fixed effect)": var,
                        "Coefficient": result.params.get(var, None),
                        "Standard Error": result.bse.get(var, None),
                        "p-value": result.pvalues.get(var, None)
                    })
                results_df = pd.DataFrame(results)
                channel_filename = os.path.join(output_dir, f"{channel}_model_results.csv")
                results_df.to_csv(channel_filename, index=False)
                print(f"Results for channel '{channel}' saved to {channel_filename}")
            except Exception as e:
                print(f"Error processing channel '{channel}': {e}")

success_columns = [
    'Planning', 'Surprise','Expectation confirmation',
    'Theory of Mind', 'Happy',
    'Frustrated'
]
random_effects = ['Dyad', 'Participant Number']
exclude_columns = ['Dyad', 'Participant Number', 'Index', 'Hunting Success', 'Potential exclude'] + success_columns
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/channel_models"
perform_linear_mixed_model_by_channel_and_save(REGRESSOR1HUNT, output_dir, success_columns, random_effects, exclude_columns)

#Correcting across tests (Hunt > Follow and Hunt > Chance) too

import numpy as np

TABLE1 = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/REGRESSOR1_ttest_results.csv")
TABLE2 = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/REGRESSOR1_HUNT_Matrix6_zeronotcorrected.csv")
TABLE1 = TABLE1[['Feature','p_value']]
TABLE1.rename(columns={'Feature':'Channel','p_value':'p'},inplace=True)
TABLE2 = TABLE2[['Channel','p']]
TABLE1['Test'] = 'huntvsfollow'
TABLE2['Test'] = 'huntvschance'
FINALTABLES = pd.concat([TABLE1,TABLE2],axis=0)
_, p_values_corrected, _, _ = multipletests(FINALTABLES['p'], method='fdr_bh')
FINALTABLES['correctedp'] = p_values_corrected
FINALTABLES['bothsurvive'] = ''
FINALTABLES['bothsurvive'] = np.where([FINALTABLES['correctedp']<0.05],'yes','no').transpose()
FINALTABLES.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/REGRESSOR1_HUNT_Matrix6_correctedacrossboth.csv")

Error processing channel 'Channel  8': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 13': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 14': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 18': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 17': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 21': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel  1': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel  2': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel  7': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel  6': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 12': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 24': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel  5': invalid syntax (<unknown>, line 1)
Error processing channel 'Channel 11':

In [ ]:
#Hypothesis 3 - Is neural synchrony significantly higher during the hunt trials compared to chance levels, and does it also show greater synchrony during the follow trials relative to chance?

import ast
import pandas as pd  
import matplotlib.pyplot as plt 
import numpy as np 
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_1samp, spearmanr, stats
from statsmodels.stats import multitest

BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
BEHAVIOURZSCORE.rename(columns={'Participant': 'Participant Number'}, inplace=True)
WTCtableold = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtable.csv")
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
WTCtableold['Dyad'] = WTCtableold['Dyad'].map(dyad_mapping)
WTCtableold.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNEW.csv")
WTCtableNULL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULL.csv")
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
WTCtableNULL['Dyad'] = WTCtableNULL['Dyad'].map(dyad_mapping)
WTCtableNULL.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLNEW.csv")

def process_and_merge_data(data_path, wtc_path,final_output_path):

    def process_data(data_path, temp_output_path):
        df = pd.read_csv(data_path)
        result_data = []
        for index, row in df.iterrows():
            channel = row['Channel']
            regions = ast.literal_eval(row['Region'])
            regions = [region.replace("45 - pars triangularis Broca's area", "46 - Dorsolateral prefrontal cortex") for region in regions]
            probabilities = ast.literal_eval(row['Probability'])
            max_prob_index = probabilities.index(max(probabilities))
            most_likely_region = regions[max_prob_index]
            result_data.append({'Channel': channel, 'Most likely region': most_likely_region})
        result_df = pd.DataFrame(result_data)
        result_df.to_csv(temp_output_path, index=False)

    def merge_and_save(wtc_path, temp_output_path, final_output_path):
        wtc_df = pd.read_csv(wtc_path)
        result_df = pd.read_csv(temp_output_path)
        merged_df = pd.merge(wtc_df, result_df, on='Channel', how='left')
        merged_df['Most likely region'] = merged_df['Most likely region'].str.replace(r'\b\d+\s*-\s*', '', regex=True)
        merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].astype(float)
        merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].fillna(0)
        if 'NULL' in wtc_path:
            grouped_df = merged_df.groupby(['Dyad','Channel','Iteration','TrialType']).agg({
                'AvgCoherenceLowFreq': 'mean',
                'AvgCoherenceMidFreq': 'mean',
                'AvgCoherenceHighFreq': 'mean',
                'Most likely region': 'first'
            }).reset_index()
            grouped_df.to_csv(final_output_path, index=False)
        else:
            grouped_df = merged_df.groupby(['Dyad','Channel','TrialType']).agg({
                'AvgCoherenceLowFreq': 'mean',
                'AvgCoherenceMidFreq': 'mean',
                'AvgCoherenceHighFreq': 'mean',
                'Most likely region': 'first'
            }).reset_index()
            grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].astype(float)
            grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].fillna(0)
            grouped_df.to_csv(final_output_path, index=False)
    temp_output_path = final_output_path.replace('.csv', '_temp.csv')
    process_data(data_path, temp_output_path)
    merge_and_save(wtc_path, temp_output_path, final_output_path)

original_data_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Brain diagrams-BA-AAL coordinates/BA-AAL regions from digitisation/fNIRStemplate-MNIAALcoordinates/FINALTABLEBAANDMNI.csv"
original_wtc_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNEW.csv"
original_final_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMES.csv"
null_data_path ="/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Brain diagrams-BA-AAL coordinates/BA-AAL regions from digitisation/fNIRStemplate-MNIAALcoordinates/FINALTABLEBAANDMNI.csv"
null_wtc_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLNEW.csv"
null_final_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMES.csv"
process_and_merge_data(original_data_path, original_wtc_path, original_final_output_path)
process_and_merge_data(null_data_path, null_wtc_path, null_final_output_path)

num_channels = 25
num_participants = 9
num_simulations = 5000  
kvals = np.arange(8)  
thvals = np.linspace(0.001, 0.2, 60) 
alpha2_threshold = 0.05  

def monte_carlo_simulation():
    kn = np.zeros((len(thvals), len(kvals)))
    for jj, th in enumerate(thvals):
        nchan = np.zeros((num_channels, num_simulations))  
        for i in range(num_simulations):
            results_array = np.random.rand(num_channels, num_participants) < th 
            nchan[:, i] = results_array.sum(axis=1) 
        kn[jj, :] = np.histogram(nchan, bins=np.arange(len(kvals) + 1), range=(0, len(kvals)))[0]
    knp = kn / (num_simulations * num_channels)  
    kth = knp < alpha2_threshold  
    allth = [np.nan] * len(kvals)
    for i in range(len(kvals)):
        max_idx = np.max(np.where(kth[:, i])[0]) if np.any(kth[:, i]) else -1
        if max_idx >= 0:
            allth[i] = thvals[max_idx]
    return allth

alpha_thresholds = monte_carlo_simulation()
df = pd.DataFrame({
    'K Threshold': kvals,
    'Alpha Threshold': alpha_thresholds
})
df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/KANDALPHATHRESHOLDS.csv")

REALDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMES.csv")
NULLDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMES.csv")

conditions = [
    ("Chase", "Low"),
    ("Chase", "Mid"),
    ("Chase", "High"),
    ("Follow", "Low"),
    ("Follow", "Mid"),
    ("Follow", "High"),
]
all_results_contrast = []
for contrast_type, freq_band in conditions:
    freq_col = f"AvgCoherence{freq_band}Freq"
    real_condition = REALDATA[REALDATA['TrialType']==contrast_type][["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA[NULLDATA['TrialType']==contrast_type][["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  
    for k_threshold in range(3, 6):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            all_results_contrast.append({
                "Condition": f"{contrast_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Channel": channel
            })
results_df_contrast = pd.DataFrame(all_results_contrast)
final_df_contrast = results_df_contrast.pivot_table(
    index=["K Threshold", "Channel"], 
    columns="Condition", 
    aggfunc=lambda x: "Significant", 
    fill_value="Not Significant"
).reset_index()
final_df_contrast.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ResultsByKThreshold.xlsx",index=False)

real_chase = REALDATA[REALDATA["TrialType"] == "Chase"]
real_follow = REALDATA[REALDATA["TrialType"] == "Follow"]
null_chase = NULLDATA[NULLDATA["TrialType"] == "Chase"]
null_follow = NULLDATA[NULLDATA["TrialType"] == "Follow"]
real_chase = real_chase.reset_index(drop=True)
real_follow = real_follow.reset_index(drop=True)
null_chase = null_chase.reset_index(drop=True)
null_follow = null_follow.reset_index(drop=True)
REALDATA_CONTRAST = real_chase.copy()
REALDATA_CONTRAST[f"AvgCoherenceLowFreq_Contrast"] = real_chase["AvgCoherenceLowFreq"] - real_follow["AvgCoherenceLowFreq"]
REALDATA_CONTRAST[f"AvgCoherenceMidFreq_Contrast"] = real_chase["AvgCoherenceMidFreq"] - real_follow["AvgCoherenceMidFreq"]
REALDATA_CONTRAST[f"AvgCoherenceHighFreq_Contrast"] = real_chase["AvgCoherenceHighFreq"] - real_follow["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST = null_chase.copy()
NULLDATA_CONTRAST[f"AvgCoherenceLowFreq_Contrast"] = null_chase["AvgCoherenceLowFreq"] - null_follow["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST[f"AvgCoherenceMidFreq_Contrast"] = null_chase["AvgCoherenceMidFreq"] - null_follow["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST[f"AvgCoherenceHighFreq_Contrast"] = null_chase["AvgCoherenceHighFreq"] - null_follow["AvgCoherenceHighFreq"]
REALDATA_CONTRAST['AvgCoherenceLowFreq_Chase'] = real_chase["AvgCoherenceLowFreq"]
REALDATA_CONTRAST['AvgCoherenceMidFreq_Chase'] = real_chase["AvgCoherenceMidFreq"]
REALDATA_CONTRAST['AvgCoherenceHighFreq_Chase'] = real_chase["AvgCoherenceHighFreq"]
REALDATA_CONTRAST['AvgCoherenceLowFreq_Follow'] = real_follow["AvgCoherenceLowFreq"]
REALDATA_CONTRAST['AvgCoherenceMidFreq_Follow'] = real_follow["AvgCoherenceMidFreq"]
REALDATA_CONTRAST['AvgCoherenceHighFreq_Follow'] = real_follow["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST['AvgCoherenceLowFreq_Chase'] = null_chase["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST['AvgCoherenceMidFreq_Chase'] = null_chase["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST['AvgCoherenceHighFreq_Chase'] = null_chase["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST['AvgCoherenceLowFreq_Follow'] = null_follow["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST['AvgCoherenceMidFreq_Follow'] = null_follow["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST['AvgCoherenceHighFreq_Follow'] = null_follow["AvgCoherenceHighFreq"]
REALDATA_CONTRAST.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")
NULLDATA_CONTRAST.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMESCONTRAST.xlsx")

conditions = [
    ("Low", "HuntMinusFollow"),
    ("Mid", "HuntMinusFollow"),
    ("High", "HuntMinusFollow")
]
all_results = []
contrast_values_list = []
for freq_band, contrast_type in conditions:
    if contrast_type == "HuntMinusFollow":
        freq_col = f"AvgCoherence{freq_band}Freq_Contrast"
    else:
        raise ValueError(f"Unknown contrast type: {contrast_type}")
    real_condition = REALDATA_CONTRAST[["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA_CONTRAST[["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  
    channel_significance = {channel: 0 for channel in real_condition["Channel"].unique()}
    for k_threshold in range(4, 5):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            channel_significance[channel] += 1
        for channel in significant_channels.index:
            channel_contrast_value = real_condition[real_condition["Channel"] == channel][freq_col].mean()
            contrast_values_list.append({
            "K Threshold": k_threshold,
            "Channel": channel,
            "Contrast Value": channel_contrast_value,
            "Frequency Band": freq_band  # Adding the frequency band
        })
            if channel_contrast_value > 0:
                direction = "Hunt > Follow"
            else:
                direction = "Follow > Hunt"
            all_results.append({
                "Condition": f"{contrast_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Direction": direction, 
                "Channel": channel
            })
significant_channels_all_thresholds = [
        channel for channel, count in channel_significance.items() if count == 1
    ]
contrast_values_df = pd.DataFrame(contrast_values_list)
thresholds_per_channel_band = contrast_values_df.groupby(['Channel', 'Frequency Band'])['K Threshold'].unique()
valid_channels = thresholds_per_channel_band[thresholds_per_channel_band.apply(lambda x: len(x) == 1)].index
contrast_values_filtered_df = contrast_values_df[
    contrast_values_df.set_index(['Channel', 'Frequency Band']).index.isin(valid_channels)
]
results_df = pd.DataFrame(all_results)
final_df = results_df.pivot_table(
    index=["K Threshold", "Channel"], 
    columns=["Condition"], 
    aggfunc=lambda x: "Significant", 
    fill_value="Not Significant"
).reset_index()
final_df['Direction'] = results_df.groupby(['K Threshold', 'Channel'])['Direction'].first().values
contrast_values_filtered_df = contrast_values_filtered_df.sort_values(
    by=['Channel', 'K Threshold', 'Frequency Band']
)
contrast_values_filtered_df['Direction'] = contrast_values_filtered_df['Contrast Value'].apply(
    lambda x: "Hunt > Follow" if x > 0 else "Follow > Hunt"
)
final_df.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ContrastResultsByKThreshold.xlsx")
contrast_values_filtered_df = contrast_values_filtered_df.sort_values(
    by=['Direction', 'Frequency Band', 'Channel'],
    ascending=[True, True, True]  
)
contrast_values_filtered_df.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Significant_Channels_Contrasts_at_all_k_thresholds.xlsx")

coherence_values_list = []
conditions = [
    ("Low", "Hunt"),
    ("Mid", "Hunt"),
    ("High", "Hunt"),
    ("Low", "Follow"),
    ("Mid", "Follow"),
    ("High", "Follow")
]
for freq_band, condition_type in conditions:
    if condition_type == "Hunt":
        freq_col = f"AvgCoherence{freq_band}Freq_Chase"
    elif condition_type == "Follow":
        freq_col = f"AvgCoherence{freq_band}Freq_Follow"
    else:
        raise ValueError(f"Unknown condition type: {condition_type}")
    real_condition = REALDATA_CONTRAST[["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA_CONTRAST[["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  # Simulate alpha thresholds (from previous code)
    channel_significance = {channel: 0 for channel in real_condition["Channel"].unique()}
    for k_threshold in range(4, 5):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            channel_coherence_value = real_condition[real_condition["Channel"] == channel][freq_col].mean()
            coherence_values_list.append({
                "K Threshold": k_threshold,
                "Channel": channel,
                f"{freq_band}Freq_{condition_type}": channel_coherence_value,
                "Frequency Band": freq_band,
                "Condition": condition_type
            })
            all_results.append({
                "Condition": f"{condition_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Channel": channel,
                "Significance": "Significant"
            })
significant_channels_all_thresholds = [
    channel for channel, count in channel_significance.items() if count == 1
]
coherence_values_df = pd.DataFrame(coherence_values_list)  # Now using coherence_values_list
thresholds_per_channel_band = coherence_values_df.groupby(['Channel', 'Frequency Band','Condition'])['K Threshold'].unique()
valid_combinations = thresholds_per_channel_band[thresholds_per_channel_band.apply(
    lambda x: set([4]).issubset(set(x))
)]
valid_combinations_df = valid_combinations.index.to_frame(index=False)
valid_combinations_df_sorted = valid_combinations_df.sort_values(
    by=['Condition', 'Frequency Band', 'Channel'],
    ascending=[True, True, True]  # Sorting all columns in ascending order
)
results_df = pd.DataFrame(all_results)
final_df2 = results_df.pivot_table(
    index=["K Threshold", "Channel"], 
    columns=["Condition"], 
    aggfunc="first",
    fill_value="Not Significant"
).reset_index()
final_df2.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/CoherenceResultsByKThreshold.xlsx")
valid_combinations_df_sorted.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Significant_Channels_Coherence_at_all_k_thresholds.xlsx") 

#Does brain-to-brain synchrony increase over time?

testcoherenceacrosstrials = pd.read_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")

def calculate_cumulative_diff(df, freq_col):
    df[f'{freq_col}_diff'] = df.groupby(['Dyad', 'Channel'])[freq_col].diff().fillna(0)
    return df

df1_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceLowFreq_Follow')
df2_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceMidFreq_Follow')
df3_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceHighFreq_Follow')
df1_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceLowFreq_Chase')
df2_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceMidFreq_Chase')
df3_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceHighFreq_Chase')
low_freq_df_follow = df1_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_Chase_diff', aggfunc='sum')
mid_freq_df_follow = df2_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_Chase_diff', aggfunc='sum')
high_freq_df_follow = df3_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_Chase_diff', aggfunc='sum')
low_freq_df_hunt = df1_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_Follow_diff', aggfunc='sum')
mid_freq_df_hunt = df2_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_Follow_diff', aggfunc='sum')
high_freq_df_hunt = df3_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_Follow_diff', aggfunc='sum')

REALDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMES.csv")
hunt_trials = REALDATA[REALDATA["TrialType"] == "Chase"]
follow_trials = REALDATA[REALDATA["TrialType"] == "Follow"]
combined_trials = pd.concat([follow_trials, hunt_trials], axis=0)
df1_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceLowFreq')
df2_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceMidFreq')
df3_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceHighFreq')
low_freq_df_combined = df1_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_diff', aggfunc='sum')
mid_freq_df_combined = df2_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_diff', aggfunc='sum')
high_freq_df_combined = df3_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_diff', aggfunc='sum')

significant_results = []

def perform_ttest_and_store(df, freq_band_name, trial_type):
    p_values = []
    t_stats = []
    mean_diffs = []
    channels = []
    for channel in df.columns:
        t_stat, p_value = stats.ttest_1samp(df[channel], 0)
        mean_diff = df[channel].mean()
        p_values.append(p_value)
        t_stats.append(t_stat)
        mean_diffs.append(mean_diff)
        channels.append(channel)
    reject, pvals_corrected = multitest.multipletests(p_values, alpha=0.05, method='fdr_bh')[:2]
    for i in range(len(channels)):
        if reject[i]: 
            significant_results.append({
                'Channel': channels[i],
                'Frequency Band': freq_band_name,
                'Trial Type': trial_type,
                'p-value': p_values[i],
                'Corrected p-value': pvals_corrected[i],
                't-stat': t_stats[i],
                'Mean Difference': mean_diffs[i]
            })

perform_ttest_and_store(low_freq_df_follow, 'LowFreq', 'Follow')
perform_ttest_and_store(mid_freq_df_follow, 'MidFreq', 'Follow')
perform_ttest_and_store(high_freq_df_follow, 'HighFreq', 'Follow')
perform_ttest_and_store(low_freq_df_hunt, 'LowFreq', 'Hunt')
perform_ttest_and_store(mid_freq_df_hunt, 'MidFreq', 'Hunt')
perform_ttest_and_store(high_freq_df_hunt, 'HighFreq', 'Hunt')
perform_ttest_and_store(low_freq_df_combined, 'LowFreq', 'both')
perform_ttest_and_store(mid_freq_df_combined, 'MidFreq', 'both')
perform_ttest_and_store(high_freq_df_combined, 'HighFreq', 'both')
significant_df = pd.DataFrame(significant_results)
significant_df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/changesynchronyovertime.csv")

#Associating hunting success and behavioural variables with brain synchrony 

import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/TESTPILOT.csv')
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
filtered_df = BEHAVIOURZSCORE[BEHAVIOURZSCORE['Dyad'].isin(new_indices)]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
filtered_df['Dyad'] = filtered_df['Dyad'].map(reversed_dyad_mapping)
filtered_df['Event_Type'] = filtered_df['Event'].str.extract(r'([A-Za-z]+)')
filtered_df['Original_Order'] = filtered_df.groupby('Event_Type')['Event'].transform(lambda x: pd.factorize(x)[0] + 1)
filtered_df['Event'] = filtered_df['Event_Type'] + ' ' + filtered_df['Original_Order'].astype(str)
filtered_df['Event'] = filtered_df['Event'].str.replace('Chase', 'Hunt', regex=False)
filtered_df = filtered_df.groupby(['Dyad', 'Event', 'Time']).first().reset_index()
hunt_counter, follow_counter = 1, 1

def rename_trials(group):
    global hunt_counter, follow_counter
    if group['TrialType'].str.contains('Chase').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Hunt {hunt_counter}' if 'Chase' in x else x)
        hunt_counter += 1
    elif group['TrialType'].str.contains('Follow').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Follow {follow_counter}' if 'Follow' in x else x)
        follow_counter += 1
    return group

dfREAL = dfREAL.groupby(['Dyad', 'Onset'], group_keys=False).apply(rename_trials)
dfREAL = dfREAL.sort_values(by=['Dyad', 'Channel', 'Onset'])  
dfREAL['Timepoint'] = dfREAL.groupby(['TrialType', 'Channel']).cumcount() + 1
filtered_df['Timepoint'] = filtered_df.groupby(['Event']).cumcount() + 1
filtered_df = filtered_df.drop(columns=['Time'])
filtered_df['EventNumber'] = filtered_df['Event'].str.extract('(\d+)', expand=False).astype(int)
filtered_df = filtered_df.sort_values(by=['Dyad', 'EventNumber', 'Timepoint'])
filtered_df = filtered_df.drop(columns=['EventNumber'])
dfREAL = dfREAL.rename(columns={'TrialType': 'Event'})
df_merged = dfREAL.merge(filtered_df, on=['Event', 'Timepoint'], how='left', suffixes=('', '_behav'))
hunt_lag = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)][['Dyad', 'Lag Score']]
follow_lag = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)][['Dyad', 'Lag Score']]
hunt_mean = hunt_lag.groupby('Dyad')['Lag Score'].mean().rename('Hunt_Lag')
follow_mean = follow_lag.groupby('Dyad')['Lag Score'].mean().rename('Follow_Lag')
lag_df = pd.merge(hunt_mean, follow_mean, on='Dyad', how='inner')
corr_value = lag_df['Hunt_Lag'].corr(lag_df['Follow_Lag'], method='pearson')
corr_df = pd.DataFrame({
    'Metric': ['Correlation between Hunt and Follow Lag Scores'],
    'Value': [corr_value]
})
corr_df.to_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/lag_correlation.csv', index=False)
median_lag = ['Lag Score']
median_other = ['Distance between self and prey', 'Distance between self and other','Scaled self facing prey', 'Scaled self facing other']
coherence_columns = ['CoherenceMid', 'CoherenceHigh']
output_base = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
channel_info = {
    1: [-41, 60, 3], 2: [-19, 72, 3], 3: [12, 74, 4], 4: [36, 66, 2],
    5: [50, 51, 1], 6: [-47, 48, 16], 7: [-29, 65, 15], 8: [-9, 72, 15],
    9: [22, 71, 16], 10: [42, 61, 11], 11: [56, 39, 14], 12: [-36, 54, 27],
    13: [-15, 66, 27], 14: [8, 68, 26], 15: [28, 65, 22], 16: [47, 49, 23],
    17: [-27, 54, 36], 18: [-9, 62, 36], 19: [14, 64, 33], 20: [34, 55, 31],
    21: [-17, 52, 43], 22: [0, 55, 40], 23: [18, 56, 41], 24: [-50, 35, 27],
    25: [40, 58, 22]
}
frontopolar_channels = {1, 2, 3, 4, 7, 8, 9, 10, 13, 14, 15, 19}
dorsolateral_channels = {5, 6, 11, 12, 16, 17, 18, 20, 21, 22, 23, 24, 25}
channel_groups = {
    'right_medial': [ch for ch in frontopolar_channels if channel_info[ch][0] > 0],
    'left_medial': [ch for ch in frontopolar_channels if channel_info[ch][0] < 0],
    'right_lateral': [ch for ch in dorsolateral_channels if channel_info[ch][0] > 0],
    'left_lateral': [ch for ch in dorsolateral_channels if channel_info[ch][0] < 0]
}

def fdr_correct_series(pvals):
    corrected = multipletests(pvals.values.flatten(), alpha=0.05, method='fdr_bh')[1]
    return pd.DataFrame(corrected.reshape(pvals.shape), index=pvals.index, columns=pvals.columns)

def run_model(df, predictors, interaction=False, level='channel'):
    betas = {}
    pvals = {}
    df = df.copy()
    if any('lag_score' in p for p in predictors):
        df = df.groupby(['Event', 'Channel']).agg({
            **{col: 'mean' for col in coherence_columns},
            **{col: 'first' for col in predictors if col not in coherence_columns}
        }).reset_index()
    if interaction:
        df['TrialType'] = df['Event'].str.extract(r'(Hunt|Follow)', expand=False)
    if level == 'roi':
        df['Region'] = df['Channel'].apply(lambda x: next((region for region, chans in channel_groups.items() if x in chans), None))
        groups = df['Region'].dropna().unique()
    else:
        groups = sorted(df['Channel'].dropna().unique())
    for group in groups:
        if level == 'roi':
            sub_df = df[df['Region'] == group]
        else:
            sub_df = df[df['Channel'] == group]
        for coh in coherence_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ " + ' + '.join([f"Q('{col}') * TrialType" for col in predictors])
                else:
                    formula = f"Q('{coh}') ~ " + ' + '.join([f"Q('{col}')" for col in predictors])
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                for pred in res.pvalues.index:
                    key = (group, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

df_hunt = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)].copy()
df_follow = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)].copy()
betas1, pvals1 = run_model(df_hunt, predictors=median_lag, level='channel')
betas2, pvals2 = run_model(df_hunt, predictors=median_other, level='channel')
table1a = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_hunt, predictors=median_lag, level='roi')
betas2, pvals2 = run_model(df_hunt, predictors=median_other, level='roi')
table1b = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_follow, predictors=median_lag, level='channel')
betas2, pvals2 = run_model(df_follow, predictors=median_other, level='channel')
table2a = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_follow, predictors=median_lag, level='roi')
betas2, pvals2 = run_model(df_follow, predictors=median_other, level='roi')
table2b = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
table3a = run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='channel')
table3b = run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='roi')
model_results = {
    "table1a": table1a,
    "table1b": table1b,
    "table2a": table2a,
    "table2b": table2b,
    "table3a": table3a,
    "table3b": table3b,
}
significance_summaries = {}
label_suffix = {
    "table1a": " (Hunt trials)",
    "table1b": " (Hunt trials)",
    "table2a": " (Follow trials)",
    "table2b": " (Follow trials)",
    "table3a": " (Both trials)",
    "table3b": " (Both trials)",
}

def run_and_combine(df, predictors, interaction=False, level='channel'):
    betas1, pvals1 = run_model(df, predictors=median_lag, interaction=interaction, level=level)
    betas2, pvals2 = run_model(df, predictors=median_other, interaction=interaction, level=level)
    betas = betas1.combine_first(betas2)
    pvals = pvals1.combine_first(pvals2)
    return betas, pvals

model_results2 = {
    "table1a": run_and_combine(df_hunt, predictors=median_lag + median_other, level='channel'),
    "table2a": run_and_combine(df_follow, predictors=median_lag + median_other, level='channel'),
    "table3a": run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='channel')
}
for key, (betas_df, pvals_df) in model_results2.items():
    corrected_df = fdr_correct_series(pvals_df)
    corrected_df.to_csv(os.path.join(output_base, f"{key}.csv"))
    predictor_to_channels = {}
    suffix = label_suffix[key]
    for predictor in corrected_df.columns:
        sig_channels = corrected_df.index[corrected_df[predictor] < 0.05].tolist()
        sig_channels_with_betas = [f"{ch} [{betas_df.loc[ch, predictor]:.3f}]" for ch in sig_channels]
        predictor_to_channels[predictor + suffix] = sig_channels_with_betas
    summary_df = pd.DataFrame([predictor_to_channels])
    significance_summaries[key] = summary_df
    summary_path = os.path.join(output_base, f"{key}_significant_channels.csv")
    summary_df.to_csv(summary_path)
all_significance_df = pd.concat(significance_summaries.values(), ignore_index=True)
all_significance_df = all_significance_df[all_significance_df.apply(lambda row: any(row.astype(bool)), axis=1)]
columns_to_drop = [col for col in all_significance_df.columns if 'Group Var' in col or 'Intercept' in col or col in ["TrialType[T.Hunt] × CoherenceHigh (Both trials)","TrialType[T.Hunt] × CoherenceMid (Both trials)"]]
filtered_df = all_significance_df.drop(columns=columns_to_drop)
filtered_df.to_csv(os.path.join(output_base, "all_significant_channels_summary.csv"), index=False)

from scipy.stats import pearsonr

def print_pearson_correlations(df):
    coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
    titles = ['Low Coherence', 'Middle Coherence', 'High Coherence']
    fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(24, 24))
    axs = axs.flatten()
    for idx, col in enumerate(coherence_cols):
        ax = axs[idx]
        valid_data = df[[col, 'Hunting Success']].dropna()
        r, p = pearsonr(valid_data[col], valid_data['Hunting Success'])
        print(f"{col} vs Hunting Success: r = {round(r, 3)}, p = {round(p, 4)}")
        ax.scatter(valid_data[col], valid_data['Hunting Success'])
        ax.set_title(titles[idx], fontsize=17, fontweight='bold')
        ax.set_xlabel('Synchrony', fontsize=17, fontweight='bold')
        ax.set_ylabel('Hunting Success', fontsize=17, fontweight='bold')
        ax.grid(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
    for ax in axs[len(coherence_cols):]: 
        fig.delaxes(ax)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs"
output_path = os.path.join(output_dir, "pearsons_correlations.png")
plt.tight_layout()
plt.savefig(output_path, dpi=300)
plt.show()
df_hunt_grouped = df_hunt.groupby('Event').agg({
    'Hunting Success': 'first',
    'CoherenceLow': 'mean',
    'CoherenceMid': 'mean',
    'CoherenceHigh': 'mean'
}).reset_index()
print_pearson_correlations(df_hunt_grouped)

results_path1 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/hunt_vs_follow_interaction_REGION_all_bands_sig.csv"
results1 = pd.read_csv(results_path1)
results_path2 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/hunt_regions_all_bands_sig.csv"
results2 = pd.read_csv(results_path2)
results_path3 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/follow_regions_all_bands_sig.csv"
results3 = pd.read_csv(results_path3)
results2['Effect'] = results2['Effect'].apply(lambda x: f"{x} (Hunt trials)")
results3['Effect'] = results3['Effect'].apply(lambda x: f"{x} (Follow trials)")
results2['Effect'] = results2['Effect'].str.replace(r"Q\('(.+?)'\)", r"\1", regex=True)
results3['Effect'] = results3['Effect'].str.replace(r"Q\('(.+?)'\)", r"\1", regex=True)
results = pd.concat([results1,results2,results3],axis=0)
base_columns = [
    'Lag Score',
    'Distance between self and other',
    'Distance between self and prey',
    'Scaled self facing prey', 
    'Scaled self facing other'
]
import re 
interaction_vars = [f"{var}:TrialType[T.Hunt]" for var in base_columns]
new_columns = ['Lag Score (Hunt trials)',
    'Lag Score (Follow trials)',
    'Distance between self and prey (Hunt trials)',
    'Distance between self and prey (Follow trials)',
    'Distance between self and other (Hunt trials)',
    'Distance between self and other (Follow trials)',
    'Scaled self facing prey (Hunt trials)',
    'Scaled self facing prey (Follow trials)',
    'Scaled self facing other (Hunt trials)',
    'Scaled self facing other (Follow trials)']
all_vars = base_columns + interaction_vars + new_columns
table_data = pd.DataFrame(index=all_vars)
frequencies = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
regions = ['right_medial', 'left_medial', 'right_lateral', 'left_lateral']
for freq in frequencies:
    for region in regions:
        subset = results[(results['Frequency'] == freq) & (results['Region'] == region)]
        col_name = f"{freq}_{region}"
        betas = subset.set_index('Effect')['Coefficient']
        cleaned_betas = betas.copy()
        cleaned_betas.index = cleaned_betas.index.to_series().apply(lambda x: re.sub(r"Q\('(.*?)'\)", r"\1", x))
        table_data[col_name] = table_data.index.map(lambda var: cleaned_betas.get(var, np.nan))
output_table_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/WTC_sig_betas_summary_table.csv"
table_data.to_csv(output_table_path)
def color_betas(val):
    if pd.isna(val):
        return ''
    color = 'red' if val > 0 else 'blue'
    return f'color: {color}'
replacements = {
    "Lag Score": "Change in alignment in movement synchrony between predators"
}
table_data.index = table_data.index.to_series().apply(
    lambda x: next((x.replace(old, new) for old, new in replacements.items() if old in x), x)
)
styled_table = table_data.style.applymap(color_betas).format("{:.4f}")
excel_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/WTC_sig_betas_summary_table.xlsx"
styled_table.to_excel(excel_output_path, engine='openpyxl')
df_hunt_sorted = df_hunt.sort_values(['Event', 'Channel']).reset_index(drop=True)
df_follow_sorted = df_follow.sort_values(['Event', 'Channel']).reset_index(drop=True)
coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
df_hunt_avg = df_hunt.groupby(['Dyad','Channel'])[coherence_cols].mean().reset_index()
df_follow_avg = df_follow.groupby(['Dyad','Channel'])[coherence_cols].mean().reset_index()
df_merged = pd.merge(df_hunt_avg, df_follow_avg, on=['Dyad','Channel'], suffixes=('_hunt', '_follow'))
df_contrast = pd.DataFrame()
df_contrast['Channel'] = df_merged['Channel']
df_contrast['Dyad'] = df_merged['Dyad']
df_contrast['CoherenceLowContrast'] = df_merged['CoherenceLow_hunt'] - df_merged['CoherenceLow_follow']
df_contrast['CoherenceMidContrast'] = df_merged['CoherenceMid_hunt'] - df_merged['CoherenceMid_follow']
df_contrast['CoherenceHighContrast'] = df_merged['CoherenceHigh_hunt'] - df_merged['CoherenceHigh_follow']
dyad_success = df_hunt.groupby('Dyad')['Hunting Success'].first().reset_index()
df_contrast = pd.merge(df_contrast, dyad_success, on='Dyad', how='left')

def run_single_predictor_model(df, predictor, interaction=False):
    betas = {}
    pvals = {}
    df = df.copy()
    df = df.groupby(['Event', 'Channel','Dyad']).agg({
        **{col: 'mean' for col in coherence_columns},
        predictor: 'first'
    }).reset_index()
    if interaction:
        df['TrialType'] = df['Event'].str.extract(r'(Hunt|Follow)', expand=False)
    for channel in sorted(df['Channel'].dropna().unique()):
        sub_df = df[df['Channel'] == channel]
        for coh in coherence_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ Q('{predictor}') * TrialType"
                else:
                    formula = f"Q('{coh}') ~ Q('{predictor}')"
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                for pred in res.pvalues.index:
                    key = (channel, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

def run_single_predictor_model2(df, predictor, interaction=False):
    betas = {}
    pvals = {}
    for channel in sorted(df['Channel'].dropna().unique()):
        sub_df = df[df['Channel'] == channel]
        for coh in coherence_contrast_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ Q('{predictor}') * TrialType"
                else:
                    formula = f"Q('{coh}') ~ Q('{predictor}')"
                    print(formula)
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                print(res.pvalues.index)
                for pred in res.pvalues.index:
                    key = (channel, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_series = pd.Series(betas)
    print(type(betas_series.index))           
    print(betas_series.index.names) 
    pval_series = pd.Series(pvals)
    print(type(pval_series.index))           
    print(pval_series.index.names) 
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

coherence_contrast_columns = ['CoherenceLowContrast', 'CoherenceMidContrast', 'CoherenceHighContrast']
betas_contrast, pvals_contrast = run_single_predictor_model2(df_contrast, 'Hunting Success', interaction=False)
df_combined = pd.concat([df_hunt_sorted, df_follow_sorted], axis=0).reset_index(drop=True)
betas_int, pvals_int = run_single_predictor_model(df_combined, 'Hunting Success', interaction=True)
betas_hunt, pvals_hunt = run_single_predictor_model(df_hunt_sorted, 'Hunting Success', interaction=False)
betas_follow, pvals_follow = run_single_predictor_model(df_follow_sorted, 'Hunting Success', interaction=False)

def fdr_correct_df(pvals_df):
    corrected_df = pd.DataFrame(index=pvals_df.index, columns=pvals_df.columns)
    for col in pvals_df.columns:
        pvals_col = pvals_df[col].dropna()
        if not pvals_col.empty:
            _, corrected_pvals, _, _ = multipletests(pvals_col, alpha=0.05, method='fdr_bh')
            corrected_df.loc[pvals_col.index, col] = corrected_pvals
    return corrected_df.astype(float)

pvals_int_corr = fdr_correct_df(pvals_int)
pvals_hunt_corr = fdr_correct_df(pvals_hunt)
pvals_follow_corr = fdr_correct_df(pvals_follow)
pvals_contrast_corr = fdr_correct_df(pvals_contrast)
cols_int = [
    "Q('Hunting Success') × CoherenceHigh",
    "Q('Hunting Success') × CoherenceMid",
    "Q('Hunting Success'):TrialType[T.Hunt] × CoherenceHigh",
    "Q('Hunting Success'):TrialType[T.Hunt] × CoherenceMid"
]
cols_simple = [
    "Q('Hunting Success') × CoherenceHigh",
    "Q('Hunting Success') × CoherenceMid"
]
coherence_int_huntfollow = [
    "Q('Hunting Success') × CoherenceHighContrast",
    "Q('Hunting Success') × CoherenceMidContrast"
]

def extract_significant_channels_and_betas(pvals_df, betas_df, cols_of_interest):
    significant_data = {}
    for col in cols_of_interest:
        if col in pvals_df.columns and col in betas_df.columns:
            sig_channels = pvals_df[pvals_df[col] < 0.05].index.tolist()
            sig_betas = betas_df.loc[sig_channels, col].tolist()
            combined = [f"{chan}: {round(beta, 4)}" for chan, beta in zip(sig_channels, sig_betas)]
            significant_data[col] = combined
    return pd.DataFrame([significant_data])

output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
sig_data_int = extract_significant_channels_and_betas(pvals_int_corr, betas_int, cols_int)
sig_data_hunt = extract_significant_channels_and_betas(pvals_hunt_corr, betas_hunt, cols_simple)
sig_data_follow = extract_significant_channels_and_betas(pvals_follow_corr, betas_follow, cols_simple)
sig_data_contrast = extract_significant_channels_and_betas(pvals_contrast_corr, betas_contrast, coherence_int_huntfollow)
sig_data_contrast.to_csv(output_path + "significant_channels_and_betas_contrast.csv")
sig_data_int.to_csv(output_path + "significant_channels_and_betas_interaction.csv")
sig_data_hunt.to_csv(output_path + "significant_channels_and_betas_hunt.csv")
sig_data_follow.to_csv(output_path + "significant_channels_and_betas_follow.csv")

In [ ]:
#Hypothesis 3 - Clipped data (to control for trial length)

import ast
import pandas as pd  
import matplotlib.pyplot as plt 
import numpy as np 
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_1samp, spearmanr, stats
from statsmodels.stats import multitest

dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
df_last8 = dfREAL.groupby(['Onset', 'Dyad', 'Channel']).filter(lambda x: len(x) >= 12).groupby(['Onset', 'Dyad', 'Channel'], group_keys=False).head(12).reset_index(drop=True)
df_last8.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattenedNEWCLIPSECOND.csv")
df_last8grouped = df_last8.groupby(['Onset', 'Dyad', 'Channel']).agg({'Duration':'first','CoherenceLow':'mean','CoherenceMid':'mean','CoherenceHigh':'mean','TrialType':'first'}).rename(columns={'CoherenceLow': 'AvgCoherenceLowFreq','CoherenceMid': 'AvgCoherenceMidFreq','CoherenceHigh': 'AvgCoherenceHighFreq'}).reset_index()
df_last8grouped.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableCLIPPEDSECOND.csv")
BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
BEHAVIOURZSCORE.rename(columns={'Participant': 'Participant Number'}, inplace=True)
WTCtableold = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableCLIPPEDSECOND.csv")
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
WTCtableold['Dyad'] = WTCtableold['Dyad'].map(dyad_mapping)
WTCtableold.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNEW.csv")
WTCtableNULL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULL.csv")
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
WTCtableNULL['Dyad'] = WTCtableNULL['Dyad'].map(dyad_mapping)
WTCtableNULL.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLNEW.csv")

def process_and_merge_data(data_path, wtc_path,final_output_path):

    def process_data(data_path, temp_output_path):
        df = pd.read_csv(data_path)
        result_data = []
        for index, row in df.iterrows():
            channel = row['Channel']
            regions = ast.literal_eval(row['Region'])
            regions = [region.replace("45 - pars triangularis Broca's area", "46 - Dorsolateral prefrontal cortex") for region in regions]
            probabilities = ast.literal_eval(row['Probability'])
            max_prob_index = probabilities.index(max(probabilities))
            most_likely_region = regions[max_prob_index]
            result_data.append({'Channel': channel, 'Most likely region': most_likely_region})
        result_df = pd.DataFrame(result_data)
        result_df.to_csv(temp_output_path, index=False)

    def merge_and_save(wtc_path, temp_output_path, final_output_path):
        wtc_df = pd.read_csv(wtc_path)
        result_df = pd.read_csv(temp_output_path)
        merged_df = pd.merge(wtc_df, result_df, on='Channel', how='left')
        merged_df['Most likely region'] = merged_df['Most likely region'].str.replace(r'\b\d+\s*-\s*', '', regex=True)
        merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].astype(float)
        merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].fillna(0)
        if 'NULL' in wtc_path:
            grouped_df = merged_df.groupby(['Dyad','Channel','Iteration','TrialType']).agg({
                'AvgCoherenceLowFreq': 'mean',
                'AvgCoherenceMidFreq': 'mean',
                'AvgCoherenceHighFreq': 'mean',
                'Most likely region': 'first'
            }).reset_index()
            grouped_df.to_csv(final_output_path, index=False)
        else:
            grouped_df = merged_df.groupby(['Dyad','Channel','TrialType']).agg({
                'AvgCoherenceLowFreq': 'mean',
                'AvgCoherenceMidFreq': 'mean',
                'AvgCoherenceHighFreq': 'mean',
                'Most likely region': 'first'
            }).reset_index()
            grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].astype(float)
            grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].fillna(0)
            grouped_df.to_csv(final_output_path, index=False)
    temp_output_path = final_output_path.replace('.csv', '_temp.csv')
    process_data(data_path, temp_output_path)
    merge_and_save(wtc_path, temp_output_path, final_output_path)

original_data_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Brain diagrams-BA-AAL coordinates/BA-AAL regions from digitisation/fNIRStemplate-MNIAALcoordinates/FINALTABLEBAANDMNI.csv"
original_wtc_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNEW.csv"
original_final_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCLIPPEDSECOND.csv"
null_data_path ="/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Brain diagrams-BA-AAL coordinates/BA-AAL regions from digitisation/fNIRStemplate-MNIAALcoordinates/FINALTABLEBAANDMNI.csv"
null_wtc_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLNEW.csv"
null_final_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMES.csv"
process_and_merge_data(original_data_path, original_wtc_path, original_final_output_path)
process_and_merge_data(null_data_path, null_wtc_path, null_final_output_path)

num_channels = 25
num_participants = 9
num_simulations = 5000  
kvals = np.arange(8)  
thvals = np.linspace(0.001, 0.2, 60) 
alpha2_threshold = 0.05  

def monte_carlo_simulation():
    kn = np.zeros((len(thvals), len(kvals)))
    for jj, th in enumerate(thvals):
        nchan = np.zeros((num_channels, num_simulations))  
        for i in range(num_simulations):
            results_array = np.random.rand(num_channels, num_participants) < th 
            nchan[:, i] = results_array.sum(axis=1) 
        kn[jj, :] = np.histogram(nchan, bins=np.arange(len(kvals) + 1), range=(0, len(kvals)))[0]
    knp = kn / (num_simulations * num_channels)  
    kth = knp < alpha2_threshold  
    allth = [np.nan] * len(kvals)
    for i in range(len(kvals)):
        max_idx = np.max(np.where(kth[:, i])[0]) if np.any(kth[:, i]) else -1
        if max_idx >= 0:
            allth[i] = thvals[max_idx]
    return allth

alpha_thresholds = monte_carlo_simulation()
df = pd.DataFrame({
    'K Threshold': kvals,
    'Alpha Threshold': alpha_thresholds
})
df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/KANDALPHATHRESHOLDS.csv")

REALDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCLIPPEDSECOND.csv")
NULLDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMESNEW.csv")

conditions = [
    ("Chase", "Low"),
    ("Chase", "Mid"),
    ("Chase", "High"),
    ("Follow", "Low"),
    ("Follow", "Mid"),
    ("Follow", "High"),
]
all_results_contrast = []
for contrast_type, freq_band in conditions:
    freq_col = f"AvgCoherence{freq_band}Freq"
    real_condition = REALDATA[REALDATA['TrialType']==contrast_type][["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA[NULLDATA['TrialType']==contrast_type][["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  
    for k_threshold in range(3, 6):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            all_results_contrast.append({
                "Condition": f"{contrast_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Channel": channel
            })
results_df_contrast = pd.DataFrame(all_results_contrast)
final_df_contrast = results_df_contrast.pivot_table(
    index=["K Threshold", "Channel"], 
    columns="Condition", 
    aggfunc=lambda x: "Significant", 
    fill_value="Not Significant"
).reset_index()
final_df_contrast.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ResultsByKThresholdNEWCLIPSECOND.xlsx",index=False)

real_chase = REALDATA[REALDATA["TrialType"] == "Chase"]
real_follow = REALDATA[REALDATA["TrialType"] == "Follow"]
null_chase = NULLDATA[NULLDATA["TrialType"] == "Chase"]
null_follow = NULLDATA[NULLDATA["TrialType"] == "Follow"]
real_chase = real_chase.reset_index(drop=True)
real_follow = real_follow.reset_index(drop=True)
null_chase = null_chase.reset_index(drop=True)
null_follow = null_follow.reset_index(drop=True)
REALDATA_CONTRAST = real_chase.copy()
REALDATA_CONTRAST[f"AvgCoherenceLowFreq_Contrast"] = real_chase["AvgCoherenceLowFreq"] - real_follow["AvgCoherenceLowFreq"]
REALDATA_CONTRAST[f"AvgCoherenceMidFreq_Contrast"] = real_chase["AvgCoherenceMidFreq"] - real_follow["AvgCoherenceMidFreq"]
REALDATA_CONTRAST[f"AvgCoherenceHighFreq_Contrast"] = real_chase["AvgCoherenceHighFreq"] - real_follow["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST = null_chase.copy()
NULLDATA_CONTRAST[f"AvgCoherenceLowFreq_Contrast"] = null_chase["AvgCoherenceLowFreq"] - null_follow["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST[f"AvgCoherenceMidFreq_Contrast"] = null_chase["AvgCoherenceMidFreq"] - null_follow["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST[f"AvgCoherenceHighFreq_Contrast"] = null_chase["AvgCoherenceHighFreq"] - null_follow["AvgCoherenceHighFreq"]
REALDATA_CONTRAST['AvgCoherenceLowFreq_Chase'] = real_chase["AvgCoherenceLowFreq"]
REALDATA_CONTRAST['AvgCoherenceMidFreq_Chase'] = real_chase["AvgCoherenceMidFreq"]
REALDATA_CONTRAST['AvgCoherenceHighFreq_Chase'] = real_chase["AvgCoherenceHighFreq"]
REALDATA_CONTRAST['AvgCoherenceLowFreq_Follow'] = real_follow["AvgCoherenceLowFreq"]
REALDATA_CONTRAST['AvgCoherenceMidFreq_Follow'] = real_follow["AvgCoherenceMidFreq"]
REALDATA_CONTRAST['AvgCoherenceHighFreq_Follow'] = real_follow["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST['AvgCoherenceLowFreq_Chase'] = null_chase["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST['AvgCoherenceMidFreq_Chase'] = null_chase["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST['AvgCoherenceHighFreq_Chase'] = null_chase["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST['AvgCoherenceLowFreq_Follow'] = null_follow["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST['AvgCoherenceMidFreq_Follow'] = null_follow["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST['AvgCoherenceHighFreq_Follow'] = null_follow["AvgCoherenceHighFreq"]
REALDATA_CONTRAST.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")
#NULLDATA_CONTRAST.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMESCONTRAST.xlsx")

conditions = [
    ("Low", "HuntMinusFollow"),
    ("Mid", "HuntMinusFollow"),
    ("High", "HuntMinusFollow")
]
all_results = []
contrast_values_list = []
for freq_band, contrast_type in conditions:
    if contrast_type == "HuntMinusFollow":
        freq_col = f"AvgCoherence{freq_band}Freq_Contrast"
    else:
        raise ValueError(f"Unknown contrast type: {contrast_type}")
    real_condition = REALDATA_CONTRAST[["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA_CONTRAST[["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  
    channel_significance = {channel: 0 for channel in real_condition["Channel"].unique()}
    for k_threshold in range(4, 5):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            channel_significance[channel] += 1
        for channel in significant_channels.index:
            channel_contrast_value = real_condition[real_condition["Channel"] == channel][freq_col].mean()
            contrast_values_list.append({
            "K Threshold": k_threshold,
            "Channel": channel,
            "Contrast Value": channel_contrast_value,
            "Frequency Band": freq_band  # Adding the frequency band
        })
            if channel_contrast_value > 0:
                direction = "Hunt > Follow"
            else:
                direction = "Follow > Hunt"
            all_results.append({
                "Condition": f"{contrast_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Direction": direction, 
                "Channel": channel
            })
significant_channels_all_thresholds = [
        channel for channel, count in channel_significance.items() if count == 1
    ]
contrast_values_df = pd.DataFrame(contrast_values_list)
thresholds_per_channel_band = contrast_values_df.groupby(['Channel', 'Frequency Band'])['K Threshold'].unique()
valid_channels = thresholds_per_channel_band[thresholds_per_channel_band.apply(lambda x: len(x) == 1)].index
contrast_values_filtered_df = contrast_values_df[
    contrast_values_df.set_index(['Channel', 'Frequency Band']).index.isin(valid_channels)
]
results_df = pd.DataFrame(all_results)
final_df = results_df.pivot_table(
    index=["K Threshold", "Channel"], 
    columns=["Condition"], 
    aggfunc=lambda x: "Significant", 
    fill_value="Not Significant"
).reset_index()
final_df['Direction'] = results_df.groupby(['K Threshold', 'Channel'])['Direction'].first().values
contrast_values_filtered_df = contrast_values_filtered_df.sort_values(
    by=['Channel', 'K Threshold', 'Frequency Band']
)
contrast_values_filtered_df['Direction'] = contrast_values_filtered_df['Contrast Value'].apply(
    lambda x: "Hunt > Follow" if x > 0 else "Follow > Hunt"
)
final_df.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ContrastResultsByKThresholdNEWCLIPSECOND.xlsx")
contrast_values_filtered_df = contrast_values_filtered_df.sort_values(
    by=['Direction', 'Frequency Band', 'Channel'],
    ascending=[True, True, True]  
)
contrast_values_filtered_df.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Significant_Channels_Contrasts_at_all_k_thresholdsNEWCLIPSECOND.xlsx")

coherence_values_list = []
conditions = [
    ("Low", "Hunt"),
    ("Mid", "Hunt"),
    ("High", "Hunt"),
    ("Low", "Follow"),
    ("Mid", "Follow"),
    ("High", "Follow")
]
for freq_band, condition_type in conditions:
    if condition_type == "Hunt":
        freq_col = f"AvgCoherence{freq_band}Freq_Chase"
    elif condition_type == "Follow":
        freq_col = f"AvgCoherence{freq_band}Freq_Follow"
    else:
        raise ValueError(f"Unknown condition type: {condition_type}")
    real_condition = REALDATA_CONTRAST[["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA_CONTRAST[["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  # Simulate alpha thresholds (from previous code)
    channel_significance = {channel: 0 for channel in real_condition["Channel"].unique()}
    for k_threshold in range(4, 5):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            channel_coherence_value = real_condition[real_condition["Channel"] == channel][freq_col].mean()
            coherence_values_list.append({
                "K Threshold": k_threshold,
                "Channel": channel,
                f"{freq_band}Freq_{condition_type}": channel_coherence_value,
                "Frequency Band": freq_band,
                "Condition": condition_type
            })
            all_results.append({
                "Condition": f"{condition_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Channel": channel,
                "Significance": "Significant"
            })
significant_channels_all_thresholds = [
    channel for channel, count in channel_significance.items() if count == 1
]
coherence_values_df = pd.DataFrame(coherence_values_list)  # Now using coherence_values_list
thresholds_per_channel_band = coherence_values_df.groupby(['Channel', 'Frequency Band','Condition'])['K Threshold'].unique()
valid_combinations = thresholds_per_channel_band[thresholds_per_channel_band.apply(
    lambda x: set([4]).issubset(set(x))
)]
valid_combinations_df = valid_combinations.index.to_frame(index=False)
valid_combinations_df_sorted = valid_combinations_df.sort_values(
    by=['Condition', 'Frequency Band', 'Channel'],
    ascending=[True, True, True]  # Sorting all columns in ascending order
)
results_df = pd.DataFrame(all_results)
final_df2 = results_df.pivot_table(
    index=["K Threshold", "Channel"], 
    columns=["Condition"], 
    aggfunc="first",
    fill_value="Not Significant"
).reset_index()
final_df2.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/CoherenceResultsByKThresholdNEWCLIPSECOND.xlsx")
valid_combinations_df_sorted.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Significant_Channels_Coherence_at_all_k_thresholdsNEWCLIPSECOND.xlsx") 

#Does brain-to-brain synchrony increase over time?

testcoherenceacrosstrials = pd.read_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")

def calculate_cumulative_diff(df, freq_col):
    df[f'{freq_col}_diff'] = df.groupby(['Dyad', 'Channel'])[freq_col].diff().fillna(0)
    return df

df1_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceLowFreq_Follow')
df2_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceMidFreq_Follow')
df3_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceHighFreq_Follow')
df1_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceLowFreq_Chase')
df2_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceMidFreq_Chase')
df3_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceHighFreq_Chase')
low_freq_df_follow = df1_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_Chase_diff', aggfunc='sum')
mid_freq_df_follow = df2_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_Chase_diff', aggfunc='sum')
high_freq_df_follow = df3_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_Chase_diff', aggfunc='sum')
low_freq_df_hunt = df1_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_Follow_diff', aggfunc='sum')
mid_freq_df_hunt = df2_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_Follow_diff', aggfunc='sum')
high_freq_df_hunt = df3_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_Follow_diff', aggfunc='sum')

REALDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCLIPPEDSECOND.csv")
hunt_trials = REALDATA[REALDATA["TrialType"] == "Chase"]
follow_trials = REALDATA[REALDATA["TrialType"] == "Follow"]
combined_trials = pd.concat([follow_trials, hunt_trials], axis=0)
df1_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceLowFreq')
df2_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceMidFreq')
df3_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceHighFreq')
low_freq_df_combined = df1_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_diff', aggfunc='sum')
mid_freq_df_combined = df2_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_diff', aggfunc='sum')
high_freq_df_combined = df3_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_diff', aggfunc='sum')

significant_results = []

def perform_ttest_and_store(df, freq_band_name, trial_type):
    p_values = []
    t_stats = []
    mean_diffs = []
    channels = []
    for channel in df.columns:
        t_stat, p_value = stats.ttest_1samp(df[channel], 0)
        mean_diff = df[channel].mean()
        p_values.append(p_value)
        t_stats.append(t_stat)
        mean_diffs.append(mean_diff)
        channels.append(channel)
    reject, pvals_corrected = multitest.multipletests(p_values, alpha=0.05, method='fdr_bh')[:2]
    for i in range(len(channels)):
        if reject[i]: 
            significant_results.append({
                'Channel': channels[i],
                'Frequency Band': freq_band_name,
                'Trial Type': trial_type,
                'p-value': p_values[i],
                'Corrected p-value': pvals_corrected[i],
                't-stat': t_stats[i],
                'Mean Difference': mean_diffs[i]
            })

perform_ttest_and_store(low_freq_df_follow, 'LowFreq', 'Follow')
perform_ttest_and_store(mid_freq_df_follow, 'MidFreq', 'Follow')
perform_ttest_and_store(high_freq_df_follow, 'HighFreq', 'Follow')
perform_ttest_and_store(low_freq_df_hunt, 'LowFreq', 'Hunt')
perform_ttest_and_store(mid_freq_df_hunt, 'MidFreq', 'Hunt')
perform_ttest_and_store(high_freq_df_hunt, 'HighFreq', 'Hunt')
perform_ttest_and_store(low_freq_df_combined, 'LowFreq', 'both')
perform_ttest_and_store(mid_freq_df_combined, 'MidFreq', 'both')
perform_ttest_and_store(high_freq_df_combined, 'HighFreq', 'both')
significant_df = pd.DataFrame(significant_results)
significant_df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/changesynchronyovertimeNEWCLIPSECOND.csv") 

#associating synchrony with behavioural variables

import os
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/TESTPILOT.csv')
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattenedNEWCLIPSECOND.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
filtered_df = BEHAVIOURZSCORE[BEHAVIOURZSCORE['Dyad'].isin(new_indices)]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
filtered_df['Dyad'] = filtered_df['Dyad'].map(reversed_dyad_mapping)
filtered_df['Event_Type'] = filtered_df['Event'].str.extract(r'([A-Za-z]+)')
filtered_df['Original_Order'] = filtered_df.groupby('Event_Type')['Event'].transform(lambda x: pd.factorize(x)[0] + 1)
filtered_df['Event'] = filtered_df['Event_Type'] + ' ' + filtered_df['Original_Order'].astype(str)
filtered_df['Event'] = filtered_df['Event'].str.replace('Chase', 'Hunt', regex=False)
filtered_df = filtered_df.groupby(['Dyad', 'Event', 'Time']).first().reset_index()
hunt_counter, follow_counter = 1, 1

def rename_trials(group):
    global hunt_counter, follow_counter
    if group['TrialType'].str.contains('Chase').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Hunt {hunt_counter}' if 'Chase' in x else x)
        hunt_counter += 1
    elif group['TrialType'].str.contains('Follow').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Follow {follow_counter}' if 'Follow' in x else x)
        follow_counter += 1
    return group

dfREAL = dfREAL.groupby(['Dyad', 'Onset'], group_keys=False).apply(rename_trials)
dfREAL = dfREAL.sort_values(by=['Dyad', 'Channel', 'Onset'])  
dfREAL['Timepoint'] = dfREAL.groupby(['TrialType', 'Channel']).cumcount() + 1
filtered_df['Timepoint'] = filtered_df.groupby(['Event']).cumcount() + 1
filtered_df = filtered_df.drop(columns=['Time'])
filtered_df['EventNumber'] = filtered_df['Event'].str.extract('(\d+)', expand=False).astype(int)
filtered_df = filtered_df.sort_values(by=['Dyad', 'EventNumber', 'Timepoint'])
filtered_df = filtered_df.drop(columns=['EventNumber'])
dfREAL = dfREAL.rename(columns={'TrialType': 'Event'})
df_merged = dfREAL.merge(filtered_df, on=['Event', 'Timepoint'], how='left', suffixes=('', '_behav'))
df_merged = df_merged.groupby('Event', group_keys=False).head(12)
print("df_merged shape:", df_merged.shape)
print("NaNs per column:\n", df_merged.isna().sum())
print("Unique Events:", df_merged['Event'].unique())
print("Dyads per Event:\n", df_merged.groupby('Event')['Dyad'].nunique())
hunt_lag = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)][['Dyad', 'Lag Score']]
follow_lag = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)][['Dyad', 'Lag Score']]
hunt_mean = hunt_lag.groupby('Dyad')['Lag Score'].mean().rename('Hunt_Lag')
follow_mean = follow_lag.groupby('Dyad')['Lag Score'].mean().rename('Follow_Lag')
lag_df = pd.merge(hunt_mean, follow_mean, on='Dyad', how='inner')
corr_value = lag_df['Hunt_Lag'].corr(lag_df['Follow_Lag'], method='pearson')
corr_df = pd.DataFrame({
    'Metric': ['Correlation between Hunt and Follow Lag Scores'],
    'Value': [corr_value]
})
corr_df.to_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/lag_correlation.csv', index=False)
median_lag = ['Lag Score']
median_other = ['Distance between self and prey', 'Distance between self and other','Scaled self facing prey', 'Scaled self facing other']
coherence_columns = ['CoherenceMid', 'CoherenceHigh']
output_base = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
channel_info = {
    1: [-41, 60, 3], 2: [-19, 72, 3], 3: [12, 74, 4], 4: [36, 66, 2],
    5: [50, 51, 1], 6: [-47, 48, 16], 7: [-29, 65, 15], 8: [-9, 72, 15],
    9: [22, 71, 16], 10: [42, 61, 11], 11: [56, 39, 14], 12: [-36, 54, 27],
    13: [-15, 66, 27], 14: [8, 68, 26], 15: [28, 65, 22], 16: [47, 49, 23],
    17: [-27, 54, 36], 18: [-9, 62, 36], 19: [14, 64, 33], 20: [34, 55, 31],
    21: [-17, 52, 43], 22: [0, 55, 40], 23: [18, 56, 41], 24: [-50, 35, 27],
    25: [40, 58, 22]
}
frontopolar_channels = {1, 2, 3, 4, 7, 8, 9, 10, 13, 14, 15, 19}
dorsolateral_channels = {5, 6, 11, 12, 16, 17, 18, 20, 21, 22, 23, 24, 25}
channel_groups = {
    'right_medial': [ch for ch in frontopolar_channels if channel_info[ch][0] > 0],
    'left_medial': [ch for ch in frontopolar_channels if channel_info[ch][0] < 0],
    'right_lateral': [ch for ch in dorsolateral_channels if channel_info[ch][0] > 0],
    'left_lateral': [ch for ch in dorsolateral_channels if channel_info[ch][0] < 0]
}

def fdr_correct_series(pvals):
    corrected = multipletests(pvals.values.flatten(), alpha=0.05, method='fdr_bh')[1]
    return pd.DataFrame(corrected.reshape(pvals.shape), index=pvals.index, columns=pvals.columns)

def run_model(df, predictors, interaction=False, level='channel'):
    betas = {}
    pvals = {}
    df = df.copy()
    if any('lag_score' in p for p in predictors):
        df = df.groupby(['Event', 'Channel']).agg({
            **{col: 'mean' for col in coherence_columns},
            **{col: 'first' for col in predictors if col not in coherence_columns}
        }).reset_index()
    if interaction:
        df['TrialType'] = df['Event'].str.extract(r'(Hunt|Follow)', expand=False)
    if level == 'roi':
        df['Region'] = df['Channel'].apply(lambda x: next((region for region, chans in channel_groups.items() if x in chans), None))
        groups = df['Region'].dropna().unique()
    else:
        groups = sorted(df['Channel'].dropna().unique())
    for group in groups:
        if level == 'roi':
            sub_df = df[df['Region'] == group]
        else:
            sub_df = df[df['Channel'] == group]
        for coh in coherence_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ " + ' + '.join([f"Q('{col}') * TrialType" for col in predictors])
                else:
                    formula = f"Q('{coh}') ~ " + ' + '.join([f"Q('{col}')" for col in predictors])
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                for pred in res.pvalues.index:
                    key = (group, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception as e:
                print(sub_df[['Dyad', coh] + predictors].isna().sum())
                print(f"Model failed for group='{group}', coherence='{coh}', predictors={predictors}")
                print("Error:", e)
                continue
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

numeric_cols = df_hunt.select_dtypes(include='number').columns
df_merged[numeric_cols] = df_merged.groupby('Dyad')[numeric_cols].transform(lambda g: g.interpolate(method='linear'))
df_hunt = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)].copy()
df_follow = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)].copy()
betas1, pvals1 = run_model(df_hunt, predictors=median_lag, level='channel')
betas2, pvals2 = run_model(df_hunt, predictors=median_other, level='channel')
table1a = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_hunt, predictors=median_lag, level='roi')
betas2, pvals2 = run_model(df_hunt, predictors=median_other, level='roi')
table1b = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_follow, predictors=median_lag, level='channel')
betas2, pvals2 = run_model(df_follow, predictors=median_other, level='channel')
table2a = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_follow, predictors=median_lag, level='roi')
betas2, pvals2 = run_model(df_follow, predictors=median_other, level='roi')
table2b = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
table3a = run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='channel')
table3b = run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='roi')
model_results = {
    "table1a": table1a,
    "table1b": table1b,
    "table2a": table2a,
    "table2b": table2b,
    "table3a": table3a,
    "table3b": table3b,
}
significance_summaries = {}
label_suffix = {
    "table1a": " (Hunt trials)",
    "table1b": " (Hunt trials)",
    "table2a": " (Follow trials)",
    "table2b": " (Follow trials)",
    "table3a": " (Both trials)",
    "table3b": " (Both trials)",
}

def run_and_combine(df, predictors, interaction=False, level='channel'):
    betas1, pvals1 = run_model(df, predictors=median_lag, interaction=interaction, level=level)
    betas2, pvals2 = run_model(df, predictors=median_other, interaction=interaction, level=level)
    betas = betas1.combine_first(betas2)
    pvals = pvals1.combine_first(pvals2)
    return betas, pvals

model_results2 = {
    "table1a": run_and_combine(df_hunt, predictors=median_lag + median_other, level='channel'),
    "table2a": run_and_combine(df_follow, predictors=median_lag + median_other, level='channel'),
    "table3a": run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='channel')
}
for key, (betas_df, pvals_df) in model_results2.items():
    corrected_df = fdr_correct_series(pvals_df)
    corrected_df.to_csv(os.path.join(output_base, f"{key}.csv"))
    predictor_to_channels = {}
    suffix = label_suffix[key]
    for predictor in corrected_df.columns:
        sig_channels = corrected_df.index[corrected_df[predictor] < 0.05].tolist()
        sig_channels_with_betas = [f"{ch} [{betas_df.loc[ch, predictor]:.3f}]" for ch in sig_channels]
        predictor_to_channels[predictor + suffix] = sig_channels_with_betas
    summary_df = pd.DataFrame([predictor_to_channels])
    significance_summaries[key] = summary_df
    summary_path = os.path.join(output_base, f"{key}_significant_channelsNEWCLIPSECOND.csv")
    summary_df.to_csv(summary_path)
all_significance_df = pd.concat(significance_summaries.values(), ignore_index=True)
all_significance_df = all_significance_df[all_significance_df.apply(lambda row: any(row.astype(bool)), axis=1)]
columns_to_drop = [col for col in all_significance_df.columns if 'Group Var' in col or 'Intercept' in col or col in ["TrialType[T.Hunt] × CoherenceHigh (Both trials)","TrialType[T.Hunt] × CoherenceMid (Both trials)"]]
filtered_df = all_significance_df.drop(columns=columns_to_drop)
filtered_df.to_csv(os.path.join(output_base, "all_significant_channels_summaryNEWCLIPSECOND.csv"), index=False)

from scipy.stats import pearsonr

def print_pearson_correlations(df):
    coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
    titles = ['Low Coherence', 'Middle Coherence', 'High Coherence']
    fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(24, 24))
    axs = axs.flatten()
    for idx, col in enumerate(coherence_cols):
        ax = axs[idx]
        valid_data = df[[col, 'Hunting Success']].dropna()
        r, p = pearsonr(valid_data[col], valid_data['Hunting Success'])
        print(f"{col} vs Hunting Success: r = {round(r, 3)}, p = {round(p, 4)}")
        ax.scatter(valid_data[col], valid_data['Hunting Success'])
        ax.set_title(titles[idx], fontsize=17, fontweight='bold')
        ax.set_xlabel('Synchrony', fontsize=17, fontweight='bold')
        ax.set_ylabel('Hunting Success', fontsize=17, fontweight='bold')
        ax.grid(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
    for ax in axs[len(coherence_cols):]: 
        fig.delaxes(ax)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs"
output_path = os.path.join(output_dir, "pearsons_correlations.png")
plt.tight_layout()
plt.savefig(output_path, dpi=300)
plt.show()
df_hunt_grouped = df_hunt.groupby('Event').agg({
    'Hunting Success': 'first',
    'CoherenceLow': 'mean',
    'CoherenceMid': 'mean',
    'CoherenceHigh': 'mean'
}).reset_index()
print_pearson_correlations(df_hunt_grouped)

results_path1 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/hunt_vs_follow_interaction_REGION_all_bands_sig.csv"
results1 = pd.read_csv(results_path1)
results_path2 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/hunt_regions_all_bands_sig.csv"
results2 = pd.read_csv(results_path2)
results_path3 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/follow_regions_all_bands_sig.csv"
results3 = pd.read_csv(results_path3)
results2['Effect'] = results2['Effect'].apply(lambda x: f"{x} (Hunt trials)")
results3['Effect'] = results3['Effect'].apply(lambda x: f"{x} (Follow trials)")
results2['Effect'] = results2['Effect'].str.replace(r"Q\('(.+?)'\)", r"\1", regex=True)
results3['Effect'] = results3['Effect'].str.replace(r"Q\('(.+?)'\)", r"\1", regex=True)
results = pd.concat([results1,results2,results3],axis=0)
base_columns = [
    'Lag Score',
    'Distance between self and other',
    'Distance between self and prey',
    'Scaled self facing prey', 
    'Scaled self facing other'
]
import re 
interaction_vars = [f"{var}:TrialType[T.Hunt]" for var in base_columns]
new_columns = ['Lag Score (Hunt trials)',
    'Lag Score (Follow trials)',
    'Distance between self and prey (Hunt trials)',
    'Distance between self and prey (Follow trials)',
    'Distance between self and other (Hunt trials)',
    'Distance between self and other (Follow trials)',
    'Scaled self facing prey (Hunt trials)',
    'Scaled self facing prey (Follow trials)',
    'Scaled self facing other (Hunt trials)',
    'Scaled self facing other (Follow trials)']
all_vars = base_columns + interaction_vars + new_columns
table_data = pd.DataFrame(index=all_vars)
frequencies = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
regions = ['right_medial', 'left_medial', 'right_lateral', 'left_lateral']
for freq in frequencies:
    for region in regions:
        subset = results[(results['Frequency'] == freq) & (results['Region'] == region)]
        col_name = f"{freq}_{region}"
        betas = subset.set_index('Effect')['Coefficient']
        cleaned_betas = betas.copy()
        cleaned_betas.index = cleaned_betas.index.to_series().apply(lambda x: re.sub(r"Q\('(.*?)'\)", r"\1", x))
        table_data[col_name] = table_data.index.map(lambda var: cleaned_betas.get(var, np.nan))
output_table_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/WTC_sig_betas_summary_tableNEWCLIPSECOND.csv"
table_data.to_csv(output_table_path)
def color_betas(val):
    if pd.isna(val):
        return ''
    color = 'red' if val > 0 else 'blue'
    return f'color: {color}'
replacements = {
    "Lag Score": "Change in alignment in movement synchrony between predators"
}
table_data.index = table_data.index.to_series().apply(
    lambda x: next((x.replace(old, new) for old, new in replacements.items() if old in x), x)
)
styled_table = table_data.style.applymap(color_betas).format("{:.4f}")
excel_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/WTC_sig_betas_summary_tableNEWCLIPSECOND.xlsx"
styled_table.to_excel(excel_output_path, engine='openpyxl')
df_hunt_sorted = df_hunt.sort_values(['Event', 'Channel']).reset_index(drop=True)
df_follow_sorted = df_follow.sort_values(['Event', 'Channel']).reset_index(drop=True)
coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
df_hunt_avg = df_hunt.groupby(['Dyad','Channel'])[coherence_cols].mean().reset_index()
df_follow_avg = df_follow.groupby(['Dyad','Channel'])[coherence_cols].mean().reset_index()
df_merged = pd.merge(df_hunt_avg, df_follow_avg, on=['Dyad','Channel'], suffixes=('_hunt', '_follow'))
df_contrast = pd.DataFrame()
df_contrast['Channel'] = df_merged['Channel']
df_contrast['Dyad'] = df_merged['Dyad']
df_contrast['CoherenceLowContrast'] = df_merged['CoherenceLow_hunt'] - df_merged['CoherenceLow_follow']
df_contrast['CoherenceMidContrast'] = df_merged['CoherenceMid_hunt'] - df_merged['CoherenceMid_follow']
df_contrast['CoherenceHighContrast'] = df_merged['CoherenceHigh_hunt'] - df_merged['CoherenceHigh_follow']
dyad_success = df_hunt.groupby('Dyad')['Hunting Success'].first().reset_index()
df_contrast = pd.merge(df_contrast, dyad_success, on='Dyad', how='left')

def run_single_predictor_model(df, predictor, interaction=False):
    betas = {}
    pvals = {}
    df = df.copy()
    df = df.groupby(['Event', 'Channel','Dyad']).agg({
        **{col: 'mean' for col in coherence_columns},
        predictor: 'first'
    }).reset_index()
    if interaction:
        df['TrialType'] = df['Event'].str.extract(r'(Hunt|Follow)', expand=False)
    for channel in sorted(df['Channel'].dropna().unique()):
        sub_df = df[df['Channel'] == channel]
        for coh in coherence_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ Q('{predictor}') * TrialType"
                else:
                    formula = f"Q('{coh}') ~ Q('{predictor}')"
                print(formula)
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                for pred in res.pvalues.index:
                    key = (channel, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

def run_single_predictor_model2(df, predictor, interaction=False):
    betas = {}
    pvals = {}
    for channel in sorted(df['Channel'].dropna().unique()):
        sub_df = df[df['Channel'] == channel]
        for coh in coherence_contrast_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ Q('{predictor}') * TrialType"
                else:
                    formula = f"Q('{coh}') ~ Q('{predictor}')"
                    print(formula)
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                print(res.pvalues.index)
                for pred in res.pvalues.index:
                    key = (channel, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_series = pd.Series(betas)
    print(type(betas_series.index))           
    print(betas_series.index.names) 
    pval_series = pd.Series(pvals)
    print(type(pval_series.index))           
    print(pval_series.index.names) 
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

coherence_contrast_columns = ['CoherenceLowContrast', 'CoherenceMidContrast', 'CoherenceHighContrast']
betas_contrast, pvals_contrast = run_single_predictor_model2(df_contrast, 'Hunting Success', interaction=False)
df_combined = pd.concat([df_hunt_sorted, df_follow_sorted], axis=0).reset_index(drop=True)
betas_int, pvals_int = run_single_predictor_model(df_combined, 'Hunting Success', interaction=True)
betas_hunt, pvals_hunt = run_single_predictor_model(df_hunt_sorted, 'Hunting Success', interaction=False)
betas_follow, pvals_follow = run_single_predictor_model(df_follow_sorted, 'Hunting Success', interaction=False)

def fdr_correct_df(pvals_df):
    corrected_df = pd.DataFrame(index=pvals_df.index, columns=pvals_df.columns)
    for col in pvals_df.columns:
        pvals_col = pvals_df[col].dropna()
        if not pvals_col.empty:
            _, corrected_pvals, _, _ = multipletests(pvals_col, alpha=0.05, method='fdr_bh')
            corrected_df.loc[pvals_col.index, col] = corrected_pvals
    return corrected_df.astype(float)

pvals_int_corr = fdr_correct_df(pvals_int)
pvals_hunt_corr = fdr_correct_df(pvals_hunt)
pvals_follow_corr = fdr_correct_df(pvals_follow)
pvals_contrast_corr = fdr_correct_df(pvals_contrast)
cols_int = [
    "Q('Hunting Success') × CoherenceHigh",
    "Q('Hunting Success') × CoherenceMid",
    "Q('Hunting Success'):TrialType[T.Hunt] × CoherenceHigh",
    "Q('Hunting Success'):TrialType[T.Hunt] × CoherenceMid"
]
cols_simple = [
    "Q('Hunting Success') × CoherenceHigh",
    "Q('Hunting Success') × CoherenceMid"
]
coherence_int_huntfollow = [
    "Q('Hunting Success') × CoherenceHighContrast",
    "Q('Hunting Success') × CoherenceMidContrast"
]

def extract_significant_channels_and_betas(pvals_df, betas_df, cols_of_interest):
    significant_data = {}
    for col in cols_of_interest:
        if col in pvals_df.columns and col in betas_df.columns:
            sig_channels = pvals_df[pvals_df[col] < 0.05].index.tolist()
            sig_betas = betas_df.loc[sig_channels, col].tolist()
            combined = [f"{chan}: {round(beta, 4)}" for chan, beta in zip(sig_channels, sig_betas)]
            significant_data[col] = combined
    return pd.DataFrame([significant_data])

output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
sig_data_int = extract_significant_channels_and_betas(pvals_int_corr, betas_int, cols_int)
sig_data_hunt = extract_significant_channels_and_betas(pvals_hunt_corr, betas_hunt, cols_simple)
sig_data_follow = extract_significant_channels_and_betas(pvals_follow_corr, betas_follow, cols_simple)
sig_data_contrast = extract_significant_channels_and_betas(pvals_contrast_corr, betas_contrast, coherence_int_huntfollow)
sig_data_contrast.to_csv(output_path + "significant_channels_and_betas_contrastNEWCLIPSECOND.csv")
sig_data_int.to_csv(output_path + "significant_channels_and_betas_interactionNEWCLIPSECOND.csv")
sig_data_hunt.to_csv(output_path + "significant_channels_and_betas_huntNEWCLIPSECOND.csv")
sig_data_follow.to_csv(output_path + "significant_channels_and_betas_followNEWCLIPSECOND.csv")

In [ ]:
#3. Hypothesis 3 - rerunning the model with time-shifted distribution (as opposed to phase-scrambling)

import ast
import pandas as pd  
import matplotlib.pyplot as plt 
import numpy as np 
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_1samp, spearmanr, stats
from statsmodels.stats import multitest

BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
BEHAVIOURZSCORE.rename(columns={'Participant': 'Participant Number'}, inplace=True)
WTCtableold = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtable.csv")
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
WTCtableold['Dyad'] = WTCtableold['Dyad'].map(dyad_mapping)
WTCtableold.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNEW.csv")
WTCtableNULL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLFINAL.csv")
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
WTCtableNULL['Dyad'] = WTCtableNULL['Dyad'].map(dyad_mapping)
WTCtableNULL.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLNEW.csv")

def process_and_merge_data(data_path, wtc_path,final_output_path):

    def process_data(data_path, temp_output_path):
        df = pd.read_csv(data_path)
        result_data = []
        for index, row in df.iterrows():
            channel = row['Channel']
            regions = ast.literal_eval(row['Region'])
            regions = [region.replace("45 - pars triangularis Broca's area", "46 - Dorsolateral prefrontal cortex") for region in regions]
            probabilities = ast.literal_eval(row['Probability'])
            max_prob_index = probabilities.index(max(probabilities))
            most_likely_region = regions[max_prob_index]
            result_data.append({'Channel': channel, 'Most likely region': most_likely_region})
        result_df = pd.DataFrame(result_data)
        result_df.to_csv(temp_output_path, index=False)

    def merge_and_save(wtc_path, temp_output_path, final_output_path):
        wtc_df = pd.read_csv(wtc_path)
        result_df = pd.read_csv(temp_output_path)
        merged_df = pd.merge(wtc_df, result_df, on='Channel', how='left')
        merged_df['Most likely region'] = merged_df['Most likely region'].str.replace(r'\b\d+\s*-\s*', '', regex=True)
        merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].astype(float)
        merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = merged_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].fillna(0)
        if 'NULL' in wtc_path:
            grouped_df = merged_df.groupby(['Dyad','Channel','Iteration','TrialType']).agg({
                'AvgCoherenceLowFreq': 'mean',
                'AvgCoherenceMidFreq': 'mean',
                'AvgCoherenceHighFreq': 'mean',
                'Most likely region': 'first'
            }).reset_index()
            grouped_df.to_csv(final_output_path, index=False)
        else:
            grouped_df = merged_df.groupby(['Dyad','Channel','TrialType']).agg({
                'AvgCoherenceLowFreq': 'mean',
                'AvgCoherenceMidFreq': 'mean',
                'AvgCoherenceHighFreq': 'mean',
                'Most likely region': 'first'
            }).reset_index()
            grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].astype(float)
            grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']] = grouped_df[['AvgCoherenceLowFreq', 'AvgCoherenceMidFreq', 'AvgCoherenceHighFreq']].fillna(0)
            grouped_df.to_csv(final_output_path, index=False)
    temp_output_path = final_output_path.replace('.csv', '_temp.csv')
    process_data(data_path, temp_output_path)
    merge_and_save(wtc_path, temp_output_path, final_output_path)

original_data_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Brain diagrams-BA-AAL coordinates/BA-AAL regions from digitisation/fNIRStemplate-MNIAALcoordinates/FINALTABLEBAANDMNI.csv"
original_wtc_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNEW.csv"
original_final_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMES.csv"
null_data_path ="/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Brain diagrams-BA-AAL coordinates/BA-AAL regions from digitisation/fNIRStemplate-MNIAALcoordinates/FINALTABLEBAANDMNI.csv"
null_wtc_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCtableNULLNEW.csv"
null_final_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMES.csv"
process_and_merge_data(original_data_path, original_wtc_path, original_final_output_path)
process_and_merge_data(null_data_path, null_wtc_path, null_final_output_path)

num_channels = 25
num_participants = 9
num_simulations = 5000  
kvals = np.arange(8)  
thvals = np.linspace(0.001, 0.2, 60) 
alpha2_threshold = 0.05  

def monte_carlo_simulation():
    kn = np.zeros((len(thvals), len(kvals)))
    for jj, th in enumerate(thvals):
        nchan = np.zeros((num_channels, num_simulations))  
        for i in range(num_simulations):
            results_array = np.random.rand(num_channels, num_participants) < th 
            nchan[:, i] = results_array.sum(axis=1) 
        kn[jj, :] = np.histogram(nchan, bins=np.arange(len(kvals) + 1), range=(0, len(kvals)))[0]
    knp = kn / (num_simulations * num_channels)  
    kth = knp < alpha2_threshold  
    allth = [np.nan] * len(kvals)
    for i in range(len(kvals)):
        max_idx = np.max(np.where(kth[:, i])[0]) if np.any(kth[:, i]) else -1
        if max_idx >= 0:
            allth[i] = thvals[max_idx]
    return allth

alpha_thresholds = monte_carlo_simulation()
df = pd.DataFrame({
    'K Threshold': kvals,
    'Alpha Threshold': alpha_thresholds
})
df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/KANDALPHATHRESHOLDS.csv")

REALDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMES.csv")
NULLDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMES.csv")

conditions = [
    ("Chase", "Low"),
    ("Chase", "Mid"),
    ("Chase", "High"),
    ("Follow", "Low"),
    ("Follow", "Mid"),
    ("Follow", "High"),
]
all_results_contrast = []
for contrast_type, freq_band in conditions:
    freq_col = f"AvgCoherence{freq_band}Freq"
    real_condition = REALDATA[REALDATA['TrialType']==contrast_type][["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA[NULLDATA['TrialType']==contrast_type][["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  
    for k_threshold in range(3, 6):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            all_results_contrast.append({
                "Condition": f"{contrast_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Channel": channel
            })
results_df_contrast = pd.DataFrame(all_results_contrast)
final_df_contrast = results_df_contrast.pivot_table(
    index=["K Threshold", "Channel"], 
    columns="Condition", 
    aggfunc=lambda x: "Significant", 
    fill_value="Not Significant"
).reset_index()
final_df_contrast.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ResultsByKThreshold.xlsx",index=False)

real_chase = REALDATA[REALDATA["TrialType"] == "Chase"]
real_follow = REALDATA[REALDATA["TrialType"] == "Follow"]
null_chase = NULLDATA[NULLDATA["TrialType"] == "Chase"]
null_follow = NULLDATA[NULLDATA["TrialType"] == "Follow"]
real_chase = real_chase.reset_index(drop=True)
real_follow = real_follow.reset_index(drop=True)
null_chase = null_chase.reset_index(drop=True)
null_follow = null_follow.reset_index(drop=True)
REALDATA_CONTRAST = real_chase.copy()
REALDATA_CONTRAST[f"AvgCoherenceLowFreq_Contrast"] = real_chase["AvgCoherenceLowFreq"] - real_follow["AvgCoherenceLowFreq"]
REALDATA_CONTRAST[f"AvgCoherenceMidFreq_Contrast"] = real_chase["AvgCoherenceMidFreq"] - real_follow["AvgCoherenceMidFreq"]
REALDATA_CONTRAST[f"AvgCoherenceHighFreq_Contrast"] = real_chase["AvgCoherenceHighFreq"] - real_follow["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST = null_chase.copy()
NULLDATA_CONTRAST[f"AvgCoherenceLowFreq_Contrast"] = null_chase["AvgCoherenceLowFreq"] - null_follow["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST[f"AvgCoherenceMidFreq_Contrast"] = null_chase["AvgCoherenceMidFreq"] - null_follow["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST[f"AvgCoherenceHighFreq_Contrast"] = null_chase["AvgCoherenceHighFreq"] - null_follow["AvgCoherenceHighFreq"]
REALDATA_CONTRAST['AvgCoherenceLowFreq_Chase'] = real_chase["AvgCoherenceLowFreq"]
REALDATA_CONTRAST['AvgCoherenceMidFreq_Chase'] = real_chase["AvgCoherenceMidFreq"]
REALDATA_CONTRAST['AvgCoherenceHighFreq_Chase'] = real_chase["AvgCoherenceHighFreq"]
REALDATA_CONTRAST['AvgCoherenceLowFreq_Follow'] = real_follow["AvgCoherenceLowFreq"]
REALDATA_CONTRAST['AvgCoherenceMidFreq_Follow'] = real_follow["AvgCoherenceMidFreq"]
REALDATA_CONTRAST['AvgCoherenceHighFreq_Follow'] = real_follow["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST['AvgCoherenceLowFreq_Chase'] = null_chase["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST['AvgCoherenceMidFreq_Chase'] = null_chase["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST['AvgCoherenceHighFreq_Chase'] = null_chase["AvgCoherenceHighFreq"]
NULLDATA_CONTRAST['AvgCoherenceLowFreq_Follow'] = null_follow["AvgCoherenceLowFreq"]
NULLDATA_CONTRAST['AvgCoherenceMidFreq_Follow'] = null_follow["AvgCoherenceMidFreq"]
NULLDATA_CONTRAST['AvgCoherenceHighFreq_Follow'] = null_follow["AvgCoherenceHighFreq"]
REALDATA_CONTRAST.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")
NULLDATA_CONTRAST.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMENULLWITHNAMESCONTRAST.xlsx")

conditions = [
    ("Low", "HuntMinusFollow"),
    ("Mid", "HuntMinusFollow"),
    ("High", "HuntMinusFollow")
]
all_results = []
for freq_band, contrast_type in conditions:
    if contrast_type == "HuntMinusFollow":
        freq_col = f"AvgCoherence{freq_band}Freq_Chase"
        freq_col2 = f"AvgCoherence{freq_band}Freq_Follow"
    else:
        raise ValueError(f"Unknown contrast type: {contrast_type}")
    real_condition = REALDATA_CONTRAST[["Dyad", "Channel", freq_col]]
    real_condition2 = REALDATA_CONTRAST[["Dyad", "Channel", freq_col2]]
    channel_names = real_condition["Channel"].unique()
    p_values = []
    tvals = []
    for channel in channel_names:
        channel_values = real_condition[real_condition["Channel"] == channel][freq_col]
        channel_values2 = real_condition2[real_condition2["Channel"] == channel][freq_col2]
        t_stat, p_val = ttest_rel(channel_values, channel_values2)
        p_values.append(p_val)
        tvals.append(t_stat)
    reject, pvals_corrected, _, _ = multitest.multipletests(p_values, alpha=0.05, method='fdr_bh')
    for i, channel in enumerate(channel_names):
        if reject[i]:
            all_results.append({
            "Channel": channel,
            "Frequency": freq_band,
            "Contrast": contrast_type,
            "t_stat": tvals[i],              
            "p_uncorrected": p_values[i],
            "p_corrected": pvals_corrected[i],
          })
significant_channels_df = pd.DataFrame(all_results)
significant_channels_df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/CONTRASTTEST.csv")

coherence_values_list = []
conditions = [
    ("Low", "Hunt"),
    ("Mid", "Hunt"),
    ("High", "Hunt"),
    ("Low", "Follow"),
    ("Mid", "Follow"),
    ("High", "Follow")
]
for freq_band, condition_type in conditions:
    if condition_type == "Hunt":
        freq_col = f"AvgCoherence{freq_band}Freq_Chase"
    elif condition_type == "Follow":
        freq_col = f"AvgCoherence{freq_band}Freq_Follow"
    else:
        raise ValueError(f"Unknown condition type: {condition_type}")
    real_condition = REALDATA_CONTRAST[["Dyad", "Channel", freq_col]]
    null_condition = NULLDATA_CONTRAST[["Dyad", "Channel", "Iteration", freq_col]]
    alpha_thresholds = monte_carlo_simulation()  # Simulate alpha thresholds (from previous code)
    channel_significance = {channel: 0 for channel in real_condition["Channel"].unique()}
    for k_threshold in range(4, 5):  
        alpha1_threshold = alpha_thresholds[k_threshold]
        WCT_threshold = null_condition[freq_col].quantile(1 - alpha1_threshold)
        significant_channels = real_condition.groupby("Channel").apply(
            lambda x: (x[freq_col] > WCT_threshold).sum()
        )
        significant_channels = significant_channels[significant_channels >= k_threshold]
        for channel in significant_channels.index:
            channel_coherence_value = real_condition[real_condition["Channel"] == channel][freq_col].mean()
            coherence_values_list.append({
                "K Threshold": k_threshold,
                "Channel": channel,
                f"{freq_band}Freq_{condition_type}": channel_coherence_value,
                "Frequency Band": freq_band,
                "Condition": condition_type
            })
            all_results.append({
                "Condition": f"{condition_type}-{freq_band}",
                "K Threshold": k_threshold,
                "Channel": channel,
                "Significance": "Significant"
            })
significant_channels_all_thresholds = [
    channel for channel, count in channel_significance.items() if count == 1
]
coherence_values_df = pd.DataFrame(coherence_values_list)  # Now using coherence_values_list
thresholds_per_channel_band = coherence_values_df.groupby(['Channel', 'Frequency Band','Condition'])['K Threshold'].unique()
valid_combinations = thresholds_per_channel_band[thresholds_per_channel_band.apply(
    lambda x: set([4]).issubset(set(x))
)]
valid_combinations_df = valid_combinations.index.to_frame(index=False)
valid_combinations_df_sorted = valid_combinations_df.sort_values(
    by=['Condition', 'Frequency Band', 'Channel'],
    ascending=[True, True, True]  # Sorting all columns in ascending order
)
results_df = pd.DataFrame(all_results)
final_df2 = results_df.pivot_table(
    index=["K Threshold", "Channel"], 
    columns=["Condition"], 
    aggfunc="first",
    fill_value="Not Significant"
).reset_index()
final_df2.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/CoherenceResultsByKThreshold.xlsx")
valid_combinations_df_sorted.to_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Significant_Channels_Coherence_at_all_k_thresholds.xlsx") 

#Does brain-to-brain synchrony increase over time?

testcoherenceacrosstrials = pd.read_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")

def calculate_cumulative_diff(df, freq_col):
    df[f'{freq_col}_diff'] = df.groupby(['Dyad', 'Channel'])[freq_col].diff().fillna(0)
    return df

df1_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceLowFreq_Follow')
df2_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceMidFreq_Follow')
df3_follow = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceHighFreq_Follow')
df1_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceLowFreq_Chase')
df2_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceMidFreq_Chase')
df3_hunt = calculate_cumulative_diff(testcoherenceacrosstrials, 'AvgCoherenceHighFreq_Chase')
low_freq_df_follow = df1_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_Chase_diff', aggfunc='sum')
mid_freq_df_follow = df2_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_Chase_diff', aggfunc='sum')
high_freq_df_follow = df3_follow.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_Chase_diff', aggfunc='sum')
low_freq_df_hunt = df1_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_Follow_diff', aggfunc='sum')
mid_freq_df_hunt = df2_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_Follow_diff', aggfunc='sum')
high_freq_df_hunt = df3_hunt.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_Follow_diff', aggfunc='sum')

REALDATA = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMES.csv")
hunt_trials = REALDATA[REALDATA["TrialType"] == "Chase"]
follow_trials = REALDATA[REALDATA["TrialType"] == "Follow"]
combined_trials = pd.concat([follow_trials, hunt_trials], axis=0)
df1_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceLowFreq')
df2_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceMidFreq')
df3_combined = calculate_cumulative_diff(combined_trials, 'AvgCoherenceHighFreq')
low_freq_df_combined = df1_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceLowFreq_diff', aggfunc='sum')
mid_freq_df_combined = df2_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceMidFreq_diff', aggfunc='sum')
high_freq_df_combined = df3_combined.pivot_table(index='Dyad', columns='Channel', values='AvgCoherenceHighFreq_diff', aggfunc='sum')

significant_results = []

def perform_ttest_and_store(df, freq_band_name, trial_type):
    p_values = []
    t_stats = []
    mean_diffs = []
    channels = []
    for channel in df.columns:
        t_stat, p_value = stats.ttest_1samp(df[channel], 0)
        mean_diff = df[channel].mean()
        p_values.append(p_value)
        t_stats.append(t_stat)
        mean_diffs.append(mean_diff)
        channels.append(channel)
    reject, pvals_corrected = multitest.multipletests(p_values, alpha=0.05, method='fdr_bh')[:2]
    for i in range(len(channels)):
        if reject[i]: 
            significant_results.append({
                'Channel': channels[i],
                'Frequency Band': freq_band_name,
                'Trial Type': trial_type,
                'p-value': p_values[i],
                'Corrected p-value': pvals_corrected[i],
                't-stat': t_stats[i],
                'Mean Difference': mean_diffs[i]
            })

perform_ttest_and_store(low_freq_df_follow, 'LowFreq', 'Follow')
perform_ttest_and_store(mid_freq_df_follow, 'MidFreq', 'Follow')
perform_ttest_and_store(high_freq_df_follow, 'HighFreq', 'Follow')
perform_ttest_and_store(low_freq_df_hunt, 'LowFreq', 'Hunt')
perform_ttest_and_store(mid_freq_df_hunt, 'MidFreq', 'Hunt')
perform_ttest_and_store(high_freq_df_hunt, 'HighFreq', 'Hunt')
perform_ttest_and_store(low_freq_df_combined, 'LowFreq', 'both')
perform_ttest_and_store(mid_freq_df_combined, 'MidFreq', 'both')
perform_ttest_and_store(high_freq_df_combined, 'HighFreq', 'both')
significant_df = pd.DataFrame(significant_results)
significant_df.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/changesynchronyovertime.csv")

#associating behavioural variables with brain synchrony

import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/TESTPILOT.csv')
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
filtered_df = BEHAVIOURZSCORE[BEHAVIOURZSCORE['Dyad'].isin(new_indices)]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
filtered_df['Dyad'] = filtered_df['Dyad'].map(reversed_dyad_mapping)
filtered_df['Event_Type'] = filtered_df['Event'].str.extract(r'([A-Za-z]+)')
filtered_df['Original_Order'] = filtered_df.groupby('Event_Type')['Event'].transform(lambda x: pd.factorize(x)[0] + 1)
filtered_df['Event'] = filtered_df['Event_Type'] + ' ' + filtered_df['Original_Order'].astype(str)
filtered_df['Event'] = filtered_df['Event'].str.replace('Chase', 'Hunt', regex=False)
filtered_df = filtered_df.groupby(['Dyad', 'Event', 'Time']).first().reset_index()
hunt_counter, follow_counter = 1, 1

def rename_trials(group):
    global hunt_counter, follow_counter
    if group['TrialType'].str.contains('Chase').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Hunt {hunt_counter}' if 'Chase' in x else x)
        hunt_counter += 1
    elif group['TrialType'].str.contains('Follow').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Follow {follow_counter}' if 'Follow' in x else x)
        follow_counter += 1
    return group

dfREAL = dfREAL.groupby(['Dyad', 'Onset'], group_keys=False).apply(rename_trials)
dfREAL = dfREAL.sort_values(by=['Dyad', 'Channel', 'Onset'])  
dfREAL['Timepoint'] = dfREAL.groupby(['TrialType', 'Channel']).cumcount() + 1
filtered_df['Timepoint'] = filtered_df.groupby(['Event']).cumcount() + 1
filtered_df = filtered_df.drop(columns=['Time'])
filtered_df['EventNumber'] = filtered_df['Event'].str.extract('(\d+)', expand=False).astype(int)
filtered_df = filtered_df.sort_values(by=['Dyad', 'EventNumber', 'Timepoint'])
filtered_df = filtered_df.drop(columns=['EventNumber'])
dfREAL = dfREAL.rename(columns={'TrialType': 'Event'})
df_merged = dfREAL.merge(filtered_df, on=['Event', 'Timepoint'], how='left', suffixes=('', '_behav'))
hunt_lag = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)][['Dyad', 'Lag Score']]
follow_lag = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)][['Dyad', 'Lag Score']]
hunt_mean = hunt_lag.groupby('Dyad')['Lag Score'].mean().rename('Hunt_Lag')
follow_mean = follow_lag.groupby('Dyad')['Lag Score'].mean().rename('Follow_Lag')
lag_df = pd.merge(hunt_mean, follow_mean, on='Dyad', how='inner')
corr_value = lag_df['Hunt_Lag'].corr(lag_df['Follow_Lag'], method='pearson')
corr_df = pd.DataFrame({
    'Metric': ['Correlation between Hunt and Follow Lag Scores'],
    'Value': [corr_value]
})
corr_df.to_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/lag_correlation.csv', index=False)
median_lag = ['Lag Score']
median_other = ['Distance between self and prey', 'Distance between self and other','Scaled self facing prey', 'Scaled self facing other']
coherence_columns = ['CoherenceMid', 'CoherenceHigh']
output_base = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
channel_info = {
    1: [-41, 60, 3], 2: [-19, 72, 3], 3: [12, 74, 4], 4: [36, 66, 2],
    5: [50, 51, 1], 6: [-47, 48, 16], 7: [-29, 65, 15], 8: [-9, 72, 15],
    9: [22, 71, 16], 10: [42, 61, 11], 11: [56, 39, 14], 12: [-36, 54, 27],
    13: [-15, 66, 27], 14: [8, 68, 26], 15: [28, 65, 22], 16: [47, 49, 23],
    17: [-27, 54, 36], 18: [-9, 62, 36], 19: [14, 64, 33], 20: [34, 55, 31],
    21: [-17, 52, 43], 22: [0, 55, 40], 23: [18, 56, 41], 24: [-50, 35, 27],
    25: [40, 58, 22]
}
frontopolar_channels = {1, 2, 3, 4, 7, 8, 9, 10, 13, 14, 15, 19}
dorsolateral_channels = {5, 6, 11, 12, 16, 17, 18, 20, 21, 22, 23, 24, 25}
channel_groups = {
    'right_medial': [ch for ch in frontopolar_channels if channel_info[ch][0] > 0],
    'left_medial': [ch for ch in frontopolar_channels if channel_info[ch][0] < 0],
    'right_lateral': [ch for ch in dorsolateral_channels if channel_info[ch][0] > 0],
    'left_lateral': [ch for ch in dorsolateral_channels if channel_info[ch][0] < 0]
}

def fdr_correct_series(pvals):
    corrected = multipletests(pvals.values.flatten(), alpha=0.05, method='fdr_bh')[1]
    return pd.DataFrame(corrected.reshape(pvals.shape), index=pvals.index, columns=pvals.columns)

def run_model(df, predictors, interaction=False, level='channel'):
    betas = {}
    pvals = {}
    df = df.copy()
    if any('lag_score' in p for p in predictors):
        df = df.groupby(['Event', 'Channel']).agg({
            **{col: 'mean' for col in coherence_columns},
            **{col: 'first' for col in predictors if col not in coherence_columns}
        }).reset_index()
    if interaction:
        df['TrialType'] = df['Event'].str.extract(r'(Hunt|Follow)', expand=False)
    if level == 'roi':
        df['Region'] = df['Channel'].apply(lambda x: next((region for region, chans in channel_groups.items() if x in chans), None))
        groups = df['Region'].dropna().unique()
    else:
        groups = sorted(df['Channel'].dropna().unique())
    for group in groups:
        if level == 'roi':
            sub_df = df[df['Region'] == group]
        else:
            sub_df = df[df['Channel'] == group]
        for coh in coherence_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ " + ' + '.join([f"Q('{col}') * TrialType" for col in predictors])
                else:
                    formula = f"Q('{coh}') ~ " + ' + '.join([f"Q('{col}')" for col in predictors])
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                for pred in res.pvalues.index:
                    key = (group, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

df_hunt = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)].copy()
df_follow = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)].copy()
betas1, pvals1 = run_model(df_hunt, predictors=median_lag, level='channel')
betas2, pvals2 = run_model(df_hunt, predictors=median_other, level='channel')
table1a = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_hunt, predictors=median_lag, level='roi')
betas2, pvals2 = run_model(df_hunt, predictors=median_other, level='roi')
table1b = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_follow, predictors=median_lag, level='channel')
betas2, pvals2 = run_model(df_follow, predictors=median_other, level='channel')
table2a = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
betas1, pvals1 = run_model(df_follow, predictors=median_lag, level='roi')
betas2, pvals2 = run_model(df_follow, predictors=median_other, level='roi')
table2b = (betas1.combine_first(betas2), pvals1.combine_first(pvals2))
table3a = run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='channel')
table3b = run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='roi')
model_results = {
    "table1a": table1a,
    "table1b": table1b,
    "table2a": table2a,
    "table2b": table2b,
    "table3a": table3a,
    "table3b": table3b,
}
significance_summaries = {}
label_suffix = {
    "table1a": " (Hunt trials)",
    "table1b": " (Hunt trials)",
    "table2a": " (Follow trials)",
    "table2b": " (Follow trials)",
    "table3a": " (Both trials)",
    "table3b": " (Both trials)",
}

def run_and_combine(df, predictors, interaction=False, level='channel'):
    betas1, pvals1 = run_model(df, predictors=median_lag, interaction=interaction, level=level)
    betas2, pvals2 = run_model(df, predictors=median_other, interaction=interaction, level=level)
    betas = betas1.combine_first(betas2)
    pvals = pvals1.combine_first(pvals2)
    return betas, pvals

model_results2 = {
    "table1a": run_and_combine(df_hunt, predictors=median_lag + median_other, level='channel'),
    "table2a": run_and_combine(df_follow, predictors=median_lag + median_other, level='channel'),
    "table3a": run_model(df_merged, predictors=median_lag + median_other, interaction=True, level='channel')
}
for key, (betas_df, pvals_df) in model_results2.items():
    corrected_df = fdr_correct_series(pvals_df)
    corrected_df.to_csv(os.path.join(output_base, f"{key}.csv"))
    predictor_to_channels = {}
    suffix = label_suffix[key]
    for predictor in corrected_df.columns:
        sig_channels = corrected_df.index[corrected_df[predictor] < 0.05].tolist()
        sig_channels_with_betas = [f"{ch} [{betas_df.loc[ch, predictor]:.3f}]" for ch in sig_channels]
        predictor_to_channels[predictor + suffix] = sig_channels_with_betas
    summary_df = pd.DataFrame([predictor_to_channels])
    significance_summaries[key] = summary_df
    summary_path = os.path.join(output_base, f"{key}_significant_channels.csv")
    summary_df.to_csv(summary_path)
all_significance_df = pd.concat(significance_summaries.values(), ignore_index=True)
all_significance_df = all_significance_df[all_significance_df.apply(lambda row: any(row.astype(bool)), axis=1)]
columns_to_drop = [col for col in all_significance_df.columns if 'Group Var' in col or 'Intercept' in col or col in ["TrialType[T.Hunt] × CoherenceHigh (Both trials)","TrialType[T.Hunt] × CoherenceMid (Both trials)"]]
filtered_df = all_significance_df.drop(columns=columns_to_drop)
filtered_df.to_csv(os.path.join(output_base, "all_significant_channels_summary.csv"), index=False)

from scipy.stats import pearsonr

def print_pearson_correlations(df):
    coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
    titles = ['Low Coherence', 'Middle Coherence', 'High Coherence']
    fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(24, 24))
    axs = axs.flatten()
    for idx, col in enumerate(coherence_cols):
        ax = axs[idx]
        valid_data = df[[col, 'Hunting Success']].dropna()
        r, p = pearsonr(valid_data[col], valid_data['Hunting Success'])
        print(f"{col} vs Hunting Success: r = {round(r, 3)}, p = {round(p, 4)}")
        ax.scatter(valid_data[col], valid_data['Hunting Success'])
        ax.set_title(titles[idx], fontsize=17, fontweight='bold')
        ax.set_xlabel('Synchrony', fontsize=17, fontweight='bold')
        ax.set_ylabel('Hunting Success', fontsize=17, fontweight='bold')
        ax.grid(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
    for ax in axs[len(coherence_cols):]: 
        fig.delaxes(ax)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs"
output_path = os.path.join(output_dir, "pearsons_correlations.png")
plt.tight_layout()
plt.savefig(output_path, dpi=300)
plt.show()
df_hunt_grouped = df_hunt.groupby('Event').agg({
    'Hunting Success': 'first',
    'CoherenceLow': 'mean',
    'CoherenceMid': 'mean',
    'CoherenceHigh': 'mean'
}).reset_index()
print_pearson_correlations(df_hunt_grouped)

results_path1 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/hunt_vs_follow_interaction_REGION_all_bands_sig.csv"
results1 = pd.read_csv(results_path1)
results_path2 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/hunt_regions_all_bands_sig.csv"
results2 = pd.read_csv(results_path2)
results_path3 = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/follow_regions_all_bands_sig.csv"
results3 = pd.read_csv(results_path3)
results2['Effect'] = results2['Effect'].apply(lambda x: f"{x} (Hunt trials)")
results3['Effect'] = results3['Effect'].apply(lambda x: f"{x} (Follow trials)")
results2['Effect'] = results2['Effect'].str.replace(r"Q\('(.+?)'\)", r"\1", regex=True)
results3['Effect'] = results3['Effect'].str.replace(r"Q\('(.+?)'\)", r"\1", regex=True)
results = pd.concat([results1,results2,results3],axis=0)
base_columns = [
    'Lag Score',
    'Distance between self and other',
    'Distance between self and prey',
    'Scaled self facing prey', 
    'Scaled self facing other'
]
import re 
interaction_vars = [f"{var}:TrialType[T.Hunt]" for var in base_columns]
new_columns = ['Lag Score (Hunt trials)',
    'Lag Score (Follow trials)',
    'Distance between self and prey (Hunt trials)',
    'Distance between self and prey (Follow trials)',
    'Distance between self and other (Hunt trials)',
    'Distance between self and other (Follow trials)',
    'Scaled self facing prey (Hunt trials)',
    'Scaled self facing prey (Follow trials)',
    'Scaled self facing other (Hunt trials)',
    'Scaled self facing other (Follow trials)']
all_vars = base_columns + interaction_vars + new_columns
table_data = pd.DataFrame(index=all_vars)
frequencies = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
regions = ['right_medial', 'left_medial', 'right_lateral', 'left_lateral']
for freq in frequencies:
    for region in regions:
        subset = results[(results['Frequency'] == freq) & (results['Region'] == region)]
        col_name = f"{freq}_{region}"
        betas = subset.set_index('Effect')['Coefficient']
        cleaned_betas = betas.copy()
        cleaned_betas.index = cleaned_betas.index.to_series().apply(lambda x: re.sub(r"Q\('(.*?)'\)", r"\1", x))
        table_data[col_name] = table_data.index.map(lambda var: cleaned_betas.get(var, np.nan))
output_table_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/WTC_sig_betas_summary_table.csv"
table_data.to_csv(output_table_path)
def color_betas(val):
    if pd.isna(val):
        return ''
    color = 'red' if val > 0 else 'blue'
    return f'color: {color}'
replacements = {
    "Lag Score": "Change in alignment in movement synchrony between predators"
}
table_data.index = table_data.index.to_series().apply(
    lambda x: next((x.replace(old, new) for old, new in replacements.items() if old in x), x)
)
styled_table = table_data.style.applymap(color_betas).format("{:.4f}")
excel_output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTCmodeloutputs/WTC_sig_betas_summary_table.xlsx"
styled_table.to_excel(excel_output_path, engine='openpyxl')
df_hunt_sorted = df_hunt.sort_values(['Event', 'Channel']).reset_index(drop=True)
df_follow_sorted = df_follow.sort_values(['Event', 'Channel']).reset_index(drop=True)
coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
df_hunt_avg = df_hunt.groupby(['Dyad','Channel'])[coherence_cols].mean().reset_index()
df_follow_avg = df_follow.groupby(['Dyad','Channel'])[coherence_cols].mean().reset_index()
df_merged = pd.merge(df_hunt_avg, df_follow_avg, on=['Dyad','Channel'], suffixes=('_hunt', '_follow'))
df_contrast = pd.DataFrame()
df_contrast['Channel'] = df_merged['Channel']
df_contrast['Dyad'] = df_merged['Dyad']
df_contrast['CoherenceLowContrast'] = df_merged['CoherenceLow_hunt'] - df_merged['CoherenceLow_follow']
df_contrast['CoherenceMidContrast'] = df_merged['CoherenceMid_hunt'] - df_merged['CoherenceMid_follow']
df_contrast['CoherenceHighContrast'] = df_merged['CoherenceHigh_hunt'] - df_merged['CoherenceHigh_follow']
dyad_success = df_hunt.groupby('Dyad')['Hunting Success'].first().reset_index()
df_contrast = pd.merge(df_contrast, dyad_success, on='Dyad', how='left')

def run_single_predictor_model(df, predictor, interaction=False):
    betas = {}
    pvals = {}
    df = df.copy()
    df = df.groupby(['Event', 'Channel','Dyad']).agg({
        **{col: 'mean' for col in coherence_columns},
        predictor: 'first'
    }).reset_index()
    if interaction:
        df['TrialType'] = df['Event'].str.extract(r'(Hunt|Follow)', expand=False)
    for channel in sorted(df['Channel'].dropna().unique()):
        sub_df = df[df['Channel'] == channel]
        for coh in coherence_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ Q('{predictor}') * TrialType"
                else:
                    formula = f"Q('{coh}') ~ Q('{predictor}')"
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                for pred in res.pvalues.index:
                    key = (channel, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

def run_single_predictor_model2(df, predictor, interaction=False):
    betas = {}
    pvals = {}
    for channel in sorted(df['Channel'].dropna().unique()):
        sub_df = df[df['Channel'] == channel]
        for coh in coherence_contrast_columns:
            try:
                if interaction:
                    formula = f"Q('{coh}') ~ Q('{predictor}') * TrialType"
                else:
                    formula = f"Q('{coh}') ~ Q('{predictor}')"
                    print(formula)
                model = smf.mixedlm(formula, sub_df, groups=sub_df['Dyad'])
                res = model.fit()
                print(res.pvalues.index)
                for pred in res.pvalues.index:
                    key = (channel, f"{pred} × {coh}")
                    pvals[key] = res.pvalues[pred]
                    betas[key] = res.params[pred]
            except Exception:
                continue
    betas_series = pd.Series(betas)
    print(type(betas_series.index))           
    print(betas_series.index.names) 
    pval_series = pd.Series(pvals)
    print(type(pval_series.index))           
    print(pval_series.index.names) 
    betas_df = pd.Series(betas).unstack()
    pvals_df = pd.Series(pvals).unstack()
    return betas_df, pvals_df

coherence_contrast_columns = ['CoherenceLowContrast', 'CoherenceMidContrast', 'CoherenceHighContrast']
betas_contrast, pvals_contrast = run_single_predictor_model2(df_contrast, 'Hunting Success', interaction=False)
df_combined = pd.concat([df_hunt_sorted, df_follow_sorted], axis=0).reset_index(drop=True)
betas_int, pvals_int = run_single_predictor_model(df_combined, 'Hunting Success', interaction=True)
betas_hunt, pvals_hunt = run_single_predictor_model(df_hunt_sorted, 'Hunting Success', interaction=False)
betas_follow, pvals_follow = run_single_predictor_model(df_follow_sorted, 'Hunting Success', interaction=False)

def fdr_correct_df(pvals_df):
    corrected_df = pd.DataFrame(index=pvals_df.index, columns=pvals_df.columns)
    for col in pvals_df.columns:
        pvals_col = pvals_df[col].dropna()
        if not pvals_col.empty:
            _, corrected_pvals, _, _ = multipletests(pvals_col, alpha=0.05, method='fdr_bh')
            corrected_df.loc[pvals_col.index, col] = corrected_pvals
    return corrected_df.astype(float)

pvals_int_corr = fdr_correct_df(pvals_int)
pvals_hunt_corr = fdr_correct_df(pvals_hunt)
pvals_follow_corr = fdr_correct_df(pvals_follow)
pvals_contrast_corr = fdr_correct_df(pvals_contrast)
cols_int = [
    "Q('Hunting Success') × CoherenceHigh",
    "Q('Hunting Success') × CoherenceMid",
    "Q('Hunting Success'):TrialType[T.Hunt] × CoherenceHigh",
    "Q('Hunting Success'):TrialType[T.Hunt] × CoherenceMid"
]
cols_simple = [
    "Q('Hunting Success') × CoherenceHigh",
    "Q('Hunting Success') × CoherenceMid"
]
coherence_int_huntfollow = [
    "Q('Hunting Success') × CoherenceHighContrast",
    "Q('Hunting Success') × CoherenceMidContrast"
]

def extract_significant_channels_and_betas(pvals_df, betas_df, cols_of_interest):
    significant_data = {}
    for col in cols_of_interest:
        if col in pvals_df.columns and col in betas_df.columns:
            sig_channels = pvals_df[pvals_df[col] < 0.05].index.tolist()
            sig_betas = betas_df.loc[sig_channels, col].tolist()
            combined = [f"{chan}: {round(beta, 4)}" for chan, beta in zip(sig_channels, sig_betas)]
            significant_data[col] = combined
    return pd.DataFrame([significant_data])

output_path = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
sig_data_int = extract_significant_channels_and_betas(pvals_int_corr, betas_int, cols_int)
sig_data_hunt = extract_significant_channels_and_betas(pvals_hunt_corr, betas_hunt, cols_simple)
sig_data_follow = extract_significant_channels_and_betas(pvals_follow_corr, betas_follow, cols_simple)
sig_data_contrast = extract_significant_channels_and_betas(pvals_contrast_corr, betas_contrast, coherence_int_huntfollow)
sig_data_contrast.to_csv(output_path + "significant_channels_and_betas_contrast.csv")
sig_data_int.to_csv(output_path + "significant_channels_and_betas_interaction.csv")
sig_data_hunt.to_csv(output_path + "significant_channels_and_betas_hunt.csv")
sig_data_follow.to_csv(output_path + "significant_channels_and_betas_follow.csv")

#t-test comparing medial vs lateral channel activity 

from scipy.stats import ttest_rel
resultsofttest = pd.DataFrame()
resultsofttest2 = pd.DataFrame()
lateralchannels = [11,5,16,10,6,1]
otherchannels = [2, 3, 4, 7, 8, 9, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23]
huntdata = [np.mean(group['CoherenceMid'][group['Channel'].isin(lateralchannels)]) for i,group in df_hunt.groupby('Dyad')]
followdata = [np.mean(group['CoherenceMid'][group['Channel'].isin(lateralchannels)]) for i,group in df_follow.groupby('Dyad')]
t,p = ttest_rel(huntdata,followdata)
huntdata2 = [np.mean(group['CoherenceMid'][group['Channel'].isin(otherchannels)]) for i,group in df_hunt.groupby('Dyad')]
followdata2 = [np.mean(group['CoherenceMid'][group['Channel'].isin(otherchannels)]) for i,group in df_follow.groupby('Dyad')]
t2,p2 = ttest_rel(huntdata2,followdata2)
resultsofttest['t'] = [t]
resultsofttest['p'] = [p]
resultsofttest2['t'] = [t2]
resultsofttest2['p'] = [p2]
resultsofttest.to_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ttestlateral.csv')
resultsofttest2.to_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/ttestmedial.csv')

dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
df_last8 = dfREAL.groupby(['Onset', 'Dyad', 'Channel']).filter(lambda x: len(x) >= 26).groupby(['Onset', 'Dyad', 'Channel'], group_keys=False).head(26).reset_index(drop=True)
df_last8.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattenedNEWCLIPSECOND.csv")
BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/TESTPILOT.csv')
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
filtered_df = BEHAVIOURZSCORE[BEHAVIOURZSCORE['Dyad'].isin(new_indices)]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
filtered_df['Dyad'] = filtered_df['Dyad'].map(reversed_dyad_mapping)
filtered_df['Event_Type'] = filtered_df['Event'].str.extract(r'([A-Za-z]+)')
filtered_df['Original_Order'] = filtered_df.groupby('Event_Type')['Event'].transform(lambda x: pd.factorize(x)[0] + 1)
filtered_df['Event'] = filtered_df['Event_Type'] + ' ' + filtered_df['Original_Order'].astype(str)
filtered_df.drop(columns=['Event_Type', 'Original_Order'], inplace=True)
filtered_df['Event'] = filtered_df['Event'].str.replace('Chase', 'Hunt', regex=False)
filtered_df = filtered_df.groupby(['Dyad', 'Event', 'Time']).first().reset_index()
dfREAL = dfREAL.sort_values(by=['Dyad', 'Onset', 'Channel'])
hunt_counter, follow_counter = 1, 1
def rename_trials(group):
    global hunt_counter, follow_counter
    if group['TrialType'].str.contains('Chase').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Hunt {hunt_counter}' if 'Chase' in x else x)
        hunt_counter += 1
    elif group['TrialType'].str.contains('Follow').any():
        group['TrialType'] = group['TrialType'].apply(lambda x: f'Follow {follow_counter}' if 'Follow' in x else x)
        follow_counter += 1
    return group
dfREAL = dfREAL.groupby(['Dyad', 'Onset'], group_keys=False).apply(rename_trials)
dfREAL = dfREAL.sort_values(by=['Dyad', 'Channel', 'Onset'])  
dfREAL['Timepoint'] = dfREAL.groupby(['TrialType', 'Channel']).cumcount() + 1
filtered_df['Timepoint'] = filtered_df.groupby(['Event']).cumcount() + 1
filtered_df = filtered_df.drop(columns=['Time'])
filtered_df['EventNumber'] = filtered_df['Event'].str.extract('(\d+)', expand=False).astype(int)
filtered_df = filtered_df.sort_values(by=['Dyad', 'EventNumber', 'Timepoint'])
filtered_df = filtered_df.drop(columns=['EventNumber'])
dfREAL = dfREAL.rename(columns={'TrialType': 'Event'})
filtered_df = filtered_df.reset_index(drop=True)
df_merged = dfREAL.merge(filtered_df, on=['Event', 'Timepoint'], how='left', suffixes=('', '_behav'))
df_merged.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/MERGEDBEHAVIOURWTCTRIALBYTRIAL.csv")
df_hunt = df_merged[df_merged['Event'].str.contains('Hunt', case=False, na=False)].copy()
df_follow = df_merged[df_merged['Event'].str.contains('Follow', case=False, na=False)].copy()
df_hunt_sorted = df_hunt.sort_values(['Event', 'Channel']).reset_index(drop=True)
df_follow_sorted = df_follow.sort_values(['Event', 'Channel']).reset_index(drop=True)
coherence_cols = ['CoherenceLow', 'CoherenceMid', 'CoherenceHigh']
df_hunt_avg = df_hunt.groupby(['Dyad','Channel','Event'])[coherence_cols].mean().reset_index()
df_follow_avg = df_follow.groupby(['Dyad','Channel','Event'])[coherence_cols].mean().reset_index()
df_merged = pd.concat([df_hunt_avg, df_follow_avg])
cols_to_keep = ['Dyad', 'Channel', 'Event','CoherenceMid', 'CoherenceHigh']
df_final = df_merged.groupby(['Dyad','Channel','Event']).agg({'Dyad':'first','Channel':'first','Event':'first','CoherenceMid':'mean','CoherenceHigh':'mean'}).reset_index(drop=True)
df_finalhere = df_final.loc[:,cols_to_keep].reset_index(drop=True)
df_finalhere = df_finalhere[df_finalhere['Event'].str.contains('Hunt')]
filtered_df = filtered_df.groupby(['Dyad','Event']).agg({'Dyad':'first','Event':'first','Distance between self and other':'mean'}).reset_index(drop=True)
df_finalhere = df_finalhere.reset_index(drop=True)
filtered_df = filtered_df.sort_values(['Dyad', 'Event']).reset_index(drop=True)
filtered_df['GlobalHunt'] = filtered_df.index + 1
filtered_df['Event'] = 'Hunt ' + filtered_df['GlobalHunt'].astype(str)
FINALFRAME = pd.merge(df_finalhere,filtered_df[['Dyad','Event','Distance between self and other']])

#rerunning the INS - neural synchrony association as a sanity check

import statsmodels.formula.api as smf
corrvalueshigh = []
pvalueshigh = []
nameshigh = []
channellisthigh = []
resultshigh = pd.DataFrame()
corrvaluesmid = []
pvaluesmid = []
namesmid = []
channellistmid = []
resultsmid = pd.DataFrame()
pvalueshighlmer = []
pvaluesmidlmer = []
betavalueshigh = []
betavaluesmid = []
channels = list(range(1, 26))
for j in range(len(channels)):
       FINALFRAMENEW = FINALFRAME[FINALFRAME['Channel']==channels[j]]
       corrhigh, phigh = spearmanr(FINALFRAMENEW['CoherenceHigh'],FINALFRAMENEW['Distance between self and other'])
       corrmid, pmid = spearmanr(FINALFRAMENEW['CoherenceMid'],FINALFRAMENEW['Distance between self and other'])
       modelhigh = smf.mixedlm("CoherenceHigh ~ Q('Distance between self and other')",FINALFRAMENEW, groups=FINALFRAMENEW['Dyad'],vc_formula={"Event": "0 + C(Event)"},re_formula="~Q('Distance between self and other')")
       res_high = modelhigh.fit()
       phighlmer = res_high.pvalues["Q('Distance between self and other')"]
       betahighlmer = res_high.params["Q('Distance between self and other')"]
       modelmed = smf.mixedlm("CoherenceMid ~ Q('Distance between self and other')",FINALFRAMENEW, groups=FINALFRAMENEW['Dyad'],vc_formula={"Event": "0 + C(Event)"},re_formula="~Q('Distance between self and other')")
       res_mid = modelmed.fit()
       pmidlmer = res_mid.pvalues["Q('Distance between self and other')"]
       betamidlmer = res_mid.params["Q('Distance between self and other')"]
       channellisthigh.append(channels[j])
       channellistmid.append(channels[j])
       nameshigh.append('CoherenceHigh')
       namesmid.append('CoherenceMid')
       corrvalueshigh.append(corrhigh)
       corrvaluesmid.append(corrmid)
       betavalueshigh.append(betahighlmer)
       betavaluesmid.append(betamidlmer)
       pvalueshigh.append(phigh)
       pvaluesmid.append(pmid)
       pvalueshighlmer.append(phighlmer)
       pvaluesmidlmer.append(pmidlmer)
resultshigh['variable'] = nameshigh
resultshigh['r'] = corrvalueshigh
resultshigh['p'] = pvalueshigh
resultshigh['plmer'] = pvalueshighlmer
reject, pvals_correctedhigh, _, _ = multitest.multipletests(resultshigh['p'], alpha=0.05, method='fdr_bh')
reject, pvals_correctedhighlmer, _, _ = multitest.multipletests(resultshigh['plmer'], alpha=0.05, method='fdr_bh')
resultshigh['pcorrect'] = pvals_correctedhigh
resultshigh['pcorrectlmer'] = pvals_correctedhighlmer
resultshigh['channel'] = channellisthigh
resultshigh['beta'] = betavalueshigh
resultshigh['significantlmer'] = np.where(resultshigh['pcorrectlmer'] < 0.05, 'yes', 'no')
resultsmid['variable'] = namesmid
resultsmid['r'] = corrvaluesmid
resultsmid['beta'] = betavaluesmid
resultsmid['p'] = pvaluesmid
resultsmid['plmer'] = pvaluesmidlmer
reject, pvals_correctedmid, _, _ = multitest.multipletests(resultsmid['p'], alpha=0.05, method='fdr_bh')
reject, pvals_correctedmidlmer, _, _ = multitest.multipletests(resultsmid['plmer'], alpha=0.05, method='fdr_bh')
resultsmid['pcorrect'] = pvals_correctedmid
resultsmid['pcorrectlmer'] = pvals_correctedmidlmer
resultsmid['channel'] = channellistmid
resultsmid['significantlmer'] = np.where(resultsmid['pcorrectlmer'] < 0.05, 'yes', 'no')
resultshigh.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/highspear.csv")
resultsmid.to_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/midspear.csv")

channelshigh = resultshigh[resultshigh['significantlmer']=='yes']['channel'].to_list()
channelsmid = resultsmid[resultsmid['significantlmer']=='yes']['channel'].to_list()
for j in range(len(channelshigh)):
    plt.figure()
    FINALFRAMENEW = FINALFRAME[FINALFRAME['Channel']==channelshigh[j]]
    plt.scatter(FINALFRAMENEW['Distance between self and other'],FINALFRAMENEW['CoherenceHigh'],color='black')
    plt.title('Channel'+' '+str(channelshigh[j])+' '+'High Frequency Band')
    plt.xlabel('Distance between predators')
    plt.ylabel('Synchrony')
    plt.savefig(f"/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Plots/{channelshigh[j]}high.png")
    plt.close()
for k in range(len(channelsmid)):
    FINALFRAMENEW2 = FINALFRAME[FINALFRAME['Channel']==channelsmid[k]]
    plt.scatter(FINALFRAMENEW2['Distance between self and other'],FINALFRAMENEW2['CoherenceMid'],color='black')
    plt.title('Channel'+' '+str(channelsmid[k])+' '+'Mid Frequency Band')
    plt.xlabel('Distance between predators')
    plt.ylabel('Synchrony')
    plt.savefig(f"/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/Plots/{channelsmid[k]}mid.png")
    plt.close()

#New model without looping over channels (Coherence ~ Distance + (Distance | Dyad) + (Distance | channel) + (Distance | trial))

model_all = smf.mixedlm("CoherenceMid ~ Q('Distance between self and other')",FINALFRAME,groups=FINALFRAME['Dyad'],vc_formula={"Channel": "0 + C(Channel)","Event": "0 + C(Event)"},re_formula="~Q('Distance between self and other')" )
res_all = model_all.fit()
print(res_all.summary())

In [ ]:
#4a. Is brain synchrony associated with debrief variables? 

import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

dfREAL = pd.read_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")
BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
dfREALwithsuccess = dfREAL.merge(BEHAVIOURZSCORE[['Dyad', 'Hunting Success','Participant']], on='Dyad')
dfREALwithsuccess.drop(columns=['Unnamed: 0', 'TrialType', 'Most likely region','Participant'], inplace=True)
dfREALwithsuccess = dfREALwithsuccess.groupby(['Dyad', 'Channel']).mean().reset_index()
BEHAVIOURZSCORE.rename(columns={'Participant': 'Participant Number'}, inplace=True)
BEHAVIOURZSCORE_unique = BEHAVIOURZSCORE[['Dyad', 'Participant Number']].drop_duplicates()
DATA1 = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Debriefsummaryscores/synchronycounts.csv')
DATA1.rename(columns={'Participant': 'Participant Number'}, inplace=True)
DATA2 = pd.merge(BEHAVIOURZSCORE_unique,DATA1,how='outer')
columns_to_fill = ['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']  # Add any other columns here
for col in columns_to_fill:
    DATA2[col] = DATA2.groupby('Dyad')[col].transform(lambda x: x.ffill().bfill())
DATA2.fillna(0,inplace=True)
DATA2 = DATA2.groupby('Dyad').first().reset_index()
DATA2.drop(columns=['Participant Number'], inplace=True)
dfREALwithsuccess = pd.merge(DATA2, dfREALwithsuccess, on='Dyad', how='inner')
df = dfREALwithsuccess  
df['Averagedemotionoriginal'] = df[['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']].mean(axis=1)
emotion_cols = ['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']
X = StandardScaler().fit_transform(df[emotion_cols])
pca = PCA(n_components=1)
df['Averagedemotion'] = pca.fit_transform(X)[:, 0] 
predictors = ['Averagedemotion']
outputs = [
    'AvgCoherenceLowFreq_Chase', 'AvgCoherenceMidFreq_Chase', 'AvgCoherenceHighFreq_Chase',
    'AvgCoherenceLowFreq_Follow', 'AvgCoherenceMidFreq_Follow', 'AvgCoherenceHighFreq_Follow',
    'Hunting Success' 
]
significant_channels = []
significant_results = []
df[predictors] = df[predictors].apply(stats.zscore)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
os.makedirs(output_dir, exist_ok=True)
channels = range(1, 26)
for output in outputs:
    q_output = f"Q('{output}')"
    for predictor in predictors:
        q_predictor = f"Q('{predictor}')"
        pvals = []
        betas = []
        for channel in channels:
            df_channel = df[df['Channel'] == channel]
            if df_channel.empty:
                pvals.append(np.nan)
                continue
            formula = f"{q_output} ~ " + ' + '.join([f"Q('{p}')" for p in predictors])
            model = smf.ols(formula, data=df_channel).fit()
            pvals.append(model.pvalues[f"Q('{predictor}')"])
            betas.append(model.params[f"Q('{predictor}')"])
        corrected_pvals = multipletests(pvals, method='fdr_bh')[1]
        safe_predictor = predictor.replace(" ", "_")
        significant_channels = [
            f"{ch}:{betas[idx]:.3f}"
            for idx, (ch, pval) in enumerate(zip(channels, corrected_pvals)) if pval < 0.05
        ]
        result_label = f"{output.replace('AvgCoherence', '').replace(' ', '')}_{safe_predictor}"
        significant_results.append({
            'output_predictor': result_label,
            'SignificantChannels': ', '.join(significant_channels)
        })
        corrected_pvals_df = pd.DataFrame({
            'Channel': channels,
            f'Corrected_pval_{safe_predictor}': corrected_pvals
        })
        corrected_pvals_csv_path = os.path.join(output_dir, f"{output}_{safe_predictor}_Corrected_PValuesOVERALLEMOTION.csv")
        corrected_pvals_df.to_csv(corrected_pvals_csv_path, index=False)
        print(f"✅ Saved corrected p-values for {output} and {predictor} to {corrected_pvals_csv_path}")
summary_df = pd.DataFrame(significant_results)
summary_path = os.path.join(output_dir, "Significant_Channel_Summary_OVERALLEMOTION.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\n📄 Saved significant channel summary to {summary_path}")

#Plotting a stacked bar chart of emotions

import pandas as pd
import matplotlib.pyplot as plt

emotion_cols = ['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']
grouped = df.groupby('Dyad')[emotion_cols].first()
grouped['Summedscore'] = grouped[emotion_cols].sum(axis=1)
ax = grouped.sort_values(by='Summedscore',ascending=True).plot(kind='bar', stacked=True, figsize=(12, 6), width=0.8, colormap='Greys')
plt.xlabel('Dyad', fontsize=17, fontweight='bold')
plt.ylabel('Shared Cognitive Events (Average)', fontsize=17, fontweight='bold')
plt.xticks(rotation=0)
plt.legend(title='Cognitive Process', title_fontproperties={'weight':'bold','size':'12'})
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/dyad_emotion_stacked_plot.png', dpi=300, bbox_inches='tight') 
plt.show()

#Plotting total number of shared cognitive processes

df['Averagedemotion'] = df[['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']].mean(axis=1)
grouped = df.groupby('Dyad')['Averagedemotion'].first()
ax = grouped.sort_values(ascending=True).plot(kind='bar', figsize=(12, 6), width=0.8,color='Grey')
plt.xlabel('Dyad', fontsize=17, fontweight='bold')
plt.ylabel('Shared Cognitive Events (Average)', fontsize=17, fontweight='bold')
plt.xticks(rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/dyad_emotion_OVERALL_plot.png', dpi=300, bbox_inches='tight') 
plt.show()

#Plotting average number of cognitive processes for each dyad (shared and non-shared)

scores = pd.read_excel('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Debriefsummaryscores/summary_scores.xlsx')
scores = scores.sort_values('Participant').reset_index(drop=True)
scores['Dyad'] = (scores['Participant'] // 2) + 1
scores.drop(columns=['Expectation confirmation'],inplace=True)
scores = scores.groupby('Dyad').agg({'Participant':"first",'Theory of Mind':"mean",'Happy':"mean",'Frustrated':"mean",'Planning':"mean",'Surprise':"mean"})
scores.reset_index(inplace=True,drop=True)
dfREAL = pd.read_excel("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/FINALWTCFRAMEREALWITHNAMESCONTRAST.xlsx")
BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
dfREALwithsuccess = dfREAL.merge(BEHAVIOURZSCORE[['Dyad', 'Hunting Success','Participant']], on='Dyad')
dfREALwithsuccess.drop(columns=['Unnamed: 0', 'TrialType', 'Most likely region','Participant'], inplace=True)
dfREALwithsuccess = dfREALwithsuccess.groupby(['Dyad', 'Channel']).mean().reset_index()
BEHAVIOURZSCORE.rename(columns={'Participant': 'Participant Number'}, inplace=True)
BEHAVIOURZSCORE_unique = BEHAVIOURZSCORE[['Dyad', 'Participant Number']].drop_duplicates()
scores.rename(columns={'Participant': 'Participant Number'}, inplace=True)
DATA2 = pd.merge(BEHAVIOURZSCORE_unique,scores,how='outer')
columns_to_fill = ['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']  # Add any other columns here
for col in columns_to_fill:
    DATA2[col] = DATA2.groupby('Dyad')[col].transform(lambda x: x.ffill().bfill())
DATA2.fillna(0,inplace=True)
DATA2 = DATA2.groupby('Dyad').first().reset_index()
DATA2.drop(columns=['Participant Number'], inplace=True)
dfREALwithsuccess = pd.merge(DATA2, dfREALwithsuccess, on='Dyad', how='inner')
df = dfREALwithsuccess
df['Averagedemotion'] = df[['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']].mean(axis=1)
grouped = df.groupby('Dyad')['Averagedemotion'].first()
ax = grouped.sort_values(ascending=True).plot(kind='bar', figsize=(12, 6), width=0.8, color='Grey')
plt.xlabel('Dyad', fontsize=17, fontweight='bold')
plt.ylabel('All Cognitive Events (Average)', fontsize=17, fontweight='bold')
plt.xticks(rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/dyad_emotion_OVERALL_plot_NOTSHARED.png', dpi=300, bbox_inches='tight') 
plt.show()

#run model predicting brain activity with nunber of datapoints (averaged for each dyad), regardless of whether they are shared or not

predictors = ['Averagedemotion']
outputs = [
    'AvgCoherenceLowFreq_Chase', 'AvgCoherenceMidFreq_Chase', 'AvgCoherenceHighFreq_Chase',
    'AvgCoherenceLowFreq_Follow', 'AvgCoherenceMidFreq_Follow', 'AvgCoherenceHighFreq_Follow',
    'Hunting Success' 
]
significant_channels = []
significant_results = []
df[predictors] = df[predictors].apply(stats.zscore)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
os.makedirs(output_dir, exist_ok=True)
channels = range(1, 26)
for output in outputs:
    q_output = f"Q('{output}')"
    for predictor in predictors:
        q_predictor = f"Q('{predictor}')"
        pvals = []
        betas = []
        for channel in channels:
            df_channel = df[df['Channel'] == channel]
            if df_channel.empty:
                pvals.append(np.nan)
                continue
            formula = f"{q_output} ~ " + ' + '.join([f"Q('{p}')" for p in predictors])
            model = smf.ols(formula, data=df_channel).fit()
            pvals.append(model.pvalues[f"Q('{predictor}')"])
            betas.append(model.params[f"Q('{predictor}')"])
        corrected_pvals = multipletests(pvals, method='fdr_bh')[1]
        print(corrected_pvals)
        safe_predictor = predictor.replace(" ", "_")
        significant_channels = [
            f"{ch}:{betas[idx]:.3f}"
            for idx, (ch, pval) in enumerate(zip(channels, corrected_pvals)) if pval < 0.05
        ]
        result_label = f"{output.replace('AvgCoherence', '').replace(' ', '')}_{safe_predictor}"
        significant_results.append({
            'output_predictor': result_label,
            'SignificantChannels': ', '.join(significant_channels)
        })
        corrected_pvals_df = pd.DataFrame({
            'Channel': channels,
            f'Corrected_pval_{safe_predictor}': corrected_pvals
        })
        corrected_pvals_csv_path = os.path.join(output_dir, f"{output}_{safe_predictor}_Corrected_PValuesOVERALLEMOTION_NONSHARED.csv")
        corrected_pvals_df.to_csv(corrected_pvals_csv_path, index=False)
        print(f"✅ Saved corrected p-values for {output} and {predictor} to {corrected_pvals_csv_path}")
summary_df = pd.DataFrame(significant_results)
summary_path = os.path.join(output_dir, "Significant_Channel_Summary_OVERALLEMOTION_NONSHARED.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\n📄 Saved significant channel summary to {summary_path}")

In [ ]:
#4b. Is brain activity associated with debrief variables at the single subject level?

import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats
import rpy2.robjects.packages as rpackages

pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dfREALbetassingle = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/BETAVALUESNEW.csv")
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
dfREALbetassingle = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/BETAVALUESNEW.csv")
mask_chase = dfREALbetassingle['Index'].str.contains('Chase\*bf\(1\)', regex=True)
mask_follow = dfREALbetassingle['Index'].str.contains('Follow\*bf\(1\)', regex=True)
df_chase = dfREALbetassingle[mask_chase].copy()
df_follow = dfREALbetassingle[mask_follow].copy()
df_chase_melted = df_chase.melt(
    id_vars=['Dyad', 'Participant Number', 'Index'],
    var_name='Channel',
    value_name='Chase*bf(1)'
)
df_follow_melted = df_follow.melt(
    id_vars=['Dyad', 'Participant Number', 'Index'],
    var_name='Channel',
    value_name='Follow*bf(1)'
)
df_chase_melted['Channel'] = df_chase_melted['Channel'].astype(str)
df_follow_melted['Channel'] = df_follow_melted['Channel'].astype(str)
df_merged = pd.merge(
    df_chase_melted,
    df_follow_melted,
    on=['Dyad', 'Participant Number', 'Channel'],
    how='inner'
)
df_merged.rename(columns={'Participant Number': 'Participant'}, inplace=True)
dfREALbetassingle = df_merged.copy()
dfREALbetassingle.drop(columns=['Index_x','Index_y'],inplace=True)
dfREALbetassingle = dfREALbetassingle[dfREALbetassingle['Dyad'].isin(new_indices)]
dfREALbetassingle['Dyad'] = dfREALbetassingle['Dyad'].map(reversed_dyad_mapping)
DATA1 = pd.read_excel('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Debriefsummaryscores/summary_scores.xlsx')
DATA1['Dyad'] = DATA1['Participant'].apply(lambda x : math.ceil(x/2))
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
DATA1 = DATA1[DATA1['Dyad'].isin(new_indices)]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
DATA1['Dyad'] = DATA1['Dyad'].map(reversed_dyad_mapping)
DATA2 = pd.merge(dfREALbetassingle,DATA1,how='inner',on=['Dyad','Participant'])
columns_to_fill = ['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']  
for col in columns_to_fill:
    DATA2[col] = DATA2.groupby('Dyad')[col].transform(lambda x: x.ffill().bfill())
DATA2.fillna(0,inplace=True)
df2 = DATA2 
df2.rename(columns={'Chase*bf(1)':'ChaseBeta','Follow*bf(1)':'FollowBeta'},inplace=True)
df2['ContrastBeta'] = df2['ChaseBeta'] - df2['FollowBeta']
df2['Emotionscoreoriginal'] = df2[['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']].mean(axis=1)
emotion_cols = ['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']
X = StandardScaler().fit_transform(df2[emotion_cols])
pca = PCA(n_components=1)
df2['Emotionscore'] = pca.fit_transform(X)[:, 0] 
predictors = ['Emotionscore']
outputs2 = ['ChaseBeta', 'FollowBeta', 'ContrastBeta'] 
significant_channels = []
significant_results2 = []
df2.drop(columns='Expectation confirmation',inplace=True)
df2[predictors] = df2[predictors].apply(stats.zscore)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
os.makedirs(output_dir, exist_ok=True)
channels = range(1, 26)
df2['Channel'] = df2['Channel'].astype(int)
from pymer4.models import Lmer
for output in outputs2:
    q_output = f"Q('{output}')"
    for predictor in predictors:
        q_predictor = f"Q('{predictor}')"
        pvals = []
        betas = []
        for channel in channels:
            df_channel = df2[df2['Channel'] == channel]
            if df_channel.empty:
                pvals.append(np.nan)
                continue
            formula = f"{output} ~ {predictor}"
            model = smf.mixedlm(formula, data=df_channel, groups=df_channel['Dyad']).fit()
            pvals.append(model.pvalues[predictor])
            betas.append(model.params[predictor])
        corrected_pvals = multipletests(pvals, method='fdr_bh')[1]
        safe_predictor = predictor.replace(" ", "_")
        significant_channels = [
            f"{ch}:{betas[idx]:.3f}"
            for idx, (ch, pval) in enumerate(zip(channels, corrected_pvals)) if pval < 0.05
        ]
        result_label = safe_predictor + output
        significant_results2.append({
            'output_predictor': result_label,
            'SignificantChannels': ', '.join(significant_channels)
        })
        corrected_pvals_df = pd.DataFrame({
            'Channel': channels,
            f'Corrected_pval_{safe_predictor}': corrected_pvals
        })
        corrected_pvals_csv_path = os.path.join(output_dir, f"{output}_{safe_predictor}_Corrected_PValuesSINGLE_ONESCORE.csv")
        corrected_pvals_df.to_csv(corrected_pvals_csv_path, index=False)
        print(f"✅ Saved corrected p-values for {output} and {predictor} to {corrected_pvals_csv_path}")
summary_df2 = pd.DataFrame(significant_results2)
summary_path = os.path.join(output_dir, "Significant_Channel_SummarySINGLE_ONESCORE.csv")
summary_df2.to_csv(summary_path, index=False)

#Looking at total number of events per comparison

significant_channels = []
significant_results2 = []
df2['Emotionsum'] = df2[['Frustrated', 'Happy', 'Planning', 'Surprise', 'Theory of Mind']].sum(axis=1)
predictors = ['Emotionsum']
df2[predictors] = df2[predictors].apply(stats.zscore)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
os.makedirs(output_dir, exist_ok=True)
channels = range(1, 26)
df2['Channel'] = df2['Channel'].astype(int)
from pymer4.models import Lmer
for output in outputs2:
    q_output = f"Q('{output}')"
    for predictor in predictors:
        q_predictor = f"Q('{predictor}')"
        pvals = []
        betas = []
        for channel in channels:
            df_channel = df2[df2['Channel'] == channel]
            if df_channel.empty:
                pvals.append(np.nan)
                continue
            formula = f"{output} ~ {predictor}"
            model = smf.mixedlm(formula, data=df_channel, groups=df_channel['Dyad']).fit()
            pvals.append(model.pvalues[predictor])
            betas.append(model.params[predictor])
        corrected_pvals = multipletests(pvals, method='fdr_bh')[1]
        safe_predictor = predictor.replace(" ", "_")
        significant_channels = [
            f"{ch}:{betas[idx]:.3f}"
            for idx, (ch, pval) in enumerate(zip(channels, corrected_pvals)) if pval < 0.05
        ]
        result_label = safe_predictor + output
        significant_results2.append({
            'output_predictor': result_label,
            'SignificantChannels': ', '.join(significant_channels)
        })
        corrected_pvals_df = pd.DataFrame({
            'Channel': channels,
            f'Corrected_pval_{safe_predictor}': corrected_pvals
        })
        corrected_pvals_csv_path = os.path.join(output_dir, f"{output}_{safe_predictor}_Corrected_PValuesSINGLE_ONESCORE_SUM.csv")
        corrected_pvals_df.to_csv(corrected_pvals_csv_path, index=False)
        print(f"✅ Saved corrected p-values for {output} and {predictor} to {corrected_pvals_csv_path}")
summary_df2 = pd.DataFrame(significant_results2)
summary_path = os.path.join(output_dir, "Significant_Channel_SummarySINGLE_ONESCORE_SUM.csv")
summary_df2.to_csv(summary_path, index=False)

In [ ]:
#5. Is hunting success associated brain activity at the single-subject level? 

import math
pairTable = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/pairTable.csv")
dfREAL = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/WTC_raw_flattened.csv")
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
dfREALbetassingle = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/BETAVALUESNEW.csv")
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
dfREALbetassingle = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/BETAVALUESNEW.csv")
mask_chase = dfREALbetassingle['Index'].str.contains('Chase\*bf\(1\)', regex=True)
mask_follow = dfREALbetassingle['Index'].str.contains('Follow\*bf\(1\)', regex=True)
df_chase = dfREALbetassingle[mask_chase].copy()
df_follow = dfREALbetassingle[mask_follow].copy()
df_chase_melted = df_chase.melt(
    id_vars=['Dyad', 'Participant Number', 'Index'],
    var_name='Channel',
    value_name='Chase*bf(1)'
)
df_follow_melted = df_follow.melt(
    id_vars=['Dyad', 'Participant Number', 'Index'],
    var_name='Channel',
    value_name='Follow*bf(1)'
)
df_chase_melted['Channel'] = df_chase_melted['Channel'].astype(str)
df_follow_melted['Channel'] = df_follow_melted['Channel'].astype(str)
df_merged = pd.merge(
    df_chase_melted,
    df_follow_melted,
    on=['Dyad', 'Participant Number', 'Channel'],
    how='inner'
)
df_merged.rename(columns={'Participant Number': 'Participant'}, inplace=True)
dfREALbetassingle = df_merged.copy()
dfREALbetassingle.drop(columns=['Index_x','Index_y'],inplace=True)
dfREALbetassingle = dfREALbetassingle[dfREALbetassingle['Dyad'].isin(new_indices)]
dfREALbetassingle['Dyad'] = dfREALbetassingle['Dyad'].map(reversed_dyad_mapping)
BEHAVIOURZSCORE = pd.read_csv('/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIXRSTUDIO.csv')
old_dyad_values = range(1, 10)
new_indices = pairTable['index'][:9]
BEHAVIOURZSCORE = BEHAVIOURZSCORE[BEHAVIOURZSCORE['Dyad'].isin(new_indices)]
dyad_mapping = dict(zip(old_dyad_values, new_indices))
reversed_dyad_mapping = {v: k for k, v in dyad_mapping.items()}
BEHAVIOURZSCORE['Dyad'] = BEHAVIOURZSCORE['Dyad'].map(reversed_dyad_mapping)
DATA3 = pd.merge(dfREALbetassingle,BEHAVIOURZSCORE,how='inner',on=['Dyad','Participant'])
DATA3.rename(columns={'Chase*bf(1)':'ChaseBeta','Follow*bf(1)':'FollowBeta'},inplace=True)
DATA3['ContrastBeta'] = DATA3['ChaseBeta'] - DATA3['FollowBeta']
DATA3.rename(columns={'Hunting Success': 'Hunting_Success'}, inplace=True)
predictors = ['Hunting_Success']
outputs2 = ['ChaseBeta', 'FollowBeta', 'ContrastBeta'] 
significant_channels = []
significant_results2 = []
from scipy import stats
import os 
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
DATA3[predictors] = DATA3[predictors].apply(stats.zscore)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
os.makedirs(output_dir, exist_ok=True)
channels = range(1, 26)
DATA3['Channel'] = DATA3['Channel'].astype(int)
DATA3 = DATA3.drop(columns=['Event', 'Unnamed: 0'], errors='ignore')
agg_dict = {}
for col in DATA3.columns:
    if col in ['Dyad', 'Participant', 'Channel']:
        continue
    elif col in ['ChaseBeta', 'FollowBeta', 'ContrastBeta']:
        agg_dict[col] = 'first'
    else:
        agg_dict[col] = 'mean'
DATA3_grouped = DATA3.groupby(['Dyad', 'Participant', 'Channel']).agg(agg_dict).reset_index()
for output in outputs2:
    for predictor in predictors:
        pvals = []
        betas = []
        for channel in channels:
            df_channel = DATA3_grouped[DATA3_grouped['Channel'] == channel]
            if df_channel.empty:
                pvals.append(np.nan)
                continue
            formula = f"{output} ~ {predictor}"
            model = smf.mixedlm(formula, data=df_channel, groups=df_channel["Dyad"])
            result = model.fit()
            pvals.append(result.pvalues[predictor])
            betas.append(result.params[predictor])
        corrected_pvals = multipletests(pvals, method='fdr_bh')[1]
        print(corrected_pvals)
        safe_predictor = predictor.replace(" ", "_")
        significant_channels = [
            f"{ch}:{betas[idx]:.3f}"
            for idx, (ch, pval) in enumerate(zip(channels, corrected_pvals)) if pval < 0.05
        ]
        result_label = safe_predictor + output
        significant_results2.append({
            'output_predictor': result_label,
            'SignificantChannels': ', '.join(significant_channels)
        })
        corrected_pvals_df = pd.DataFrame({
            'Channel': channels,
            f'Corrected_pval_{safe_predictor}': corrected_pvals
        })
        corrected_pvals_csv_path = os.path.join(output_dir, f"{output}_{safe_predictor}_Corrected_PValuesSINGLEHUNTSUCCESS.csv")
        corrected_pvals_df.to_csv(corrected_pvals_csv_path, index=False)
        print(f"✅ Saved corrected p-values for {output} and {predictor} to {corrected_pvals_csv_path}")
summary_df3 = pd.DataFrame(significant_results2)
summary_path = os.path.join(output_dir, "Significant_Channel_SummarySINGLEHUNTSUCCESS.csv")
summary_df3.to_csv(summary_path, index=False)

#are behavioural variables associated with hunting success?

DATA3.rename(columns={
    'Distance between self and other': 'Distance_Self_Other',
    'Distance between self and prey': 'Distance_Self_Prey',
    'Lag Score': 'Lag_Score',
    'Heart Rate self': 'HeartRate_Self',
    'Breathing Rate self': 'BreathingRate_Self',
    'Scaled self facing other': 'Facing_Other',
    'Scaled self facing prey': 'Facing_Prey',
    'Time': 'Time'
}, inplace=True)
predictors = [
    'Distance_Self_Other',
    'Distance_Self_Prey',
    'Lag_Score',
    'HeartRate_Self',
    'BreathingRate_Self',
    'Facing_Other',
    'Facing_Prey',
    'Time'
]
outputs2 = ['ChaseBeta', 'FollowBeta', 'ContrastBeta'] 
significant_channels = []
significant_results2 = []
from scipy import stats
import os 
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
DATA3[predictors] = DATA3[predictors].apply(stats.zscore)
output_dir = "/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/"
os.makedirs(output_dir, exist_ok=True)
channels = range(1, 26)
DATA3['Channel'] = DATA3['Channel'].astype(int)
DATA3 = DATA3.drop(columns=['Event', 'Unnamed: 0'], errors='ignore')
agg_dict = {}
for col in DATA3.columns:
    if col in ['Dyad', 'Participant', 'Channel']:
        continue
    elif col in ['ChaseBeta', 'FollowBeta', 'ContrastBeta']:
        agg_dict[col] = 'first'
    else:
        agg_dict[col] = 'mean'
DATA3_grouped = DATA3.groupby(['Dyad', 'Participant', 'Channel']).agg(agg_dict).reset_index()
for output in outputs2:
    for predictor in predictors:
        pvals = []
        betas = []
        for channel in channels:
            df_channel = DATA3_grouped[DATA3_grouped['Channel'] == channel]
            if df_channel.empty:
                pvals.append(np.nan)
                continue
            formula = f"{output} ~ {predictor}"
            model = smf.mixedlm(formula, data=df_channel, groups=df_channel["Dyad"])
            result = model.fit()
            pvals.append(result.pvalues[predictor])
            betas.append(result.params[predictor])
        corrected_pvals = multipletests(pvals, method='fdr_bh')[1]
        safe_predictor = predictor.replace(" ", "_")
        significant_channels = [
            f"{ch}:{betas[idx]:.3f}"
            for idx, (ch, pval) in enumerate(zip(channels, corrected_pvals)) if pval < 0.05
        ]
        result_label = safe_predictor + output
        significant_results2.append({
            'output_predictor': result_label,
            'SignificantChannels': ', '.join(significant_channels)
        })
        corrected_pvals_df = pd.DataFrame({
            'Channel': channels,
            f'Corrected_pval_{safe_predictor}': corrected_pvals
        })
        corrected_pvals_csv_path = os.path.join(output_dir, f"{output}_{safe_predictor}_Corrected_PValuesSINGLEHUNTVARIABLES.csv")
        corrected_pvals_df.to_csv(corrected_pvals_csv_path, index=False)
        print(f"✅ Saved corrected p-values for {output} and {predictor} to {corrected_pvals_csv_path}")
summary_df3 = pd.DataFrame(significant_results2)
summary_path = os.path.join(output_dir, "Significant_Channel_SummarySINGLEHUNTVARIABLES.csv")
summary_df3.to_csv(summary_path, index=False)

In [ ]:
#6.Relevant plots for manuscript 

#Plot beta values for manuscript

import pandas as pd

Newframe = pd.DataFrame()
BETAS = pd.read_csv("/Users/macbook/Desktop/UCL PhD Work/fNIRSfMRIminecraft/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSanalysisoutputs/BETAVALUESDISTANCEANDDIRECTIONMAIN.csv")
BETASHUNT = BETAS[BETAS['Index'] == 'Sn(1) DirectionToPreyHunt']['12'].reset_index(drop=True)
Newframe['Hunt'] = BETASHUNT
BETASFOLLOW = BETAS[BETAS['Index'] == 'Sn(1) DirectionToPreyFollow']['12'].reset_index(drop=True)
Newframe['Follow'] = BETASFOLLOW
BETASCONTRAST = BETASHUNT-BETASFOLLOW
Newframe['Hunt > Follow'] = BETASCONTRAST

import matplotlib.pyplot as plt 
from scipy.stats import sem

names = ['Hunt','Follow','Hunt > Follow']
values = [BETASHUNT.mean(), BETASFOLLOW.mean(), BETASCONTRAST.mean()]
ax = plt.subplot()
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [sem(BETASHUNT),sem(BETASFOLLOW),sem(BETASCONTRAST)]
ax.errorbar(names,values,sevalues,color='black',ls='none')

BETASHUNT.mean()

#Plot heading direction differences for manuscript

plt.figure()
ax = plt.subplot()
TEST = pd.read_csv('/Users/macbook/Desktop/BETAVALUESFRONTLEFTRIGHT.csv')
frontvalue = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyfrontpredhunt']['12']
rightvalue = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyleftpredhunt']['12']
leftvalue = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyrightpredhunt']['12']
frontmean = frontvalue.mean()
leftmean = leftvalue.mean()
rightmean = rightvalue.mean()
frontSE = sem(frontvalue)
leftSE = sem(leftvalue)
rightSE = sem(rightvalue)
names = ['Ahead','Left','Right']
values = [frontmean, leftmean, rightmean]
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [frontSE,leftSE,rightSE]
ax.errorbar(names,values,sevalues,color='black',ls='none')

#Plot heading direction differences for manuscript

plt.figure()
ax = plt.subplot()
TEST = pd.read_csv('/Users/macbook/Desktop/BETAVALUES3b.csv')
block1 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockfirsthunt']['12']
block2 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblocksecondhunt']['12']
block3 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockthirdhunt']['12']
block4 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockfourthhunt']['12']
block1mean = block1.mean()
block2mean = block2.mean()
block3mean = block3.mean()
block4mean = block4.mean()
block1SE = sem(block1)
block2SE = sem(block2)
block3SE = sem(block3)
block4SE = sem(block4)
names = ['Very aligned','Aligned','Misaligned','Very misaligned']
values = [block1mean, block2mean, block3mean, block4mean]
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [block1SE,block2SE,block3SE,block4SE]
ax.errorbar(names,values,sevalues,color='black',ls='none')

#Plot heading direction differences for manuscript

plt.figure()
ax = plt.subplot()
TEST = pd.read_csv('/Users/macbook/Desktop/BETAVALUES4b.csv')
block1 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockfirsthunt']['12']
block2 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblocksecondhunt']['12']
block1mean = block1.mean()
block2mean = block2.mean()
block1SE = sem(block1)
block2SE = sem(block2)
names = ['Aligned (0-90)','Misaligned (90-180)']
values = [block1mean, block2mean]
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [block1SE,block2SE]
ax.errorbar(names,values,sevalues,color='black',ls='none')

#Plot heading direction differences for manuscript

plt.figure()
ax = plt.subplot()
TEST = pd.read_csv('/Users/macbook/Desktop/BETAVALUES5b.csv')
block1 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockfirsthunt']['12']
block2 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblocksecondhunt']['12']
block3 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockthirdhunt']['12']
block4 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockfourthhunt']['12']
block5 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblockfifthhunt']['12']
block6 = TEST[['12','Participant Number','Index']][TEST['Index']=='Sn(1) preyblocksixthhunt']['12']
block1mean = block1.mean()
block2mean = block2.mean()
block3mean = block3.mean()
block4mean = block4.mean()
block5mean = block5.mean()
block6mean = block6.mean()
block1SE = sem(block1)
block2SE = sem(block2)
block3SE = sem(block3)
block4SE = sem(block4)
block5SE = sem(block5)
block6SE = sem(block6)
names = ['0-30','30-60','60-90','90-120','120-150','150-180']
values = [block1mean, block2mean, block3mean, block4mean, block5mean, block6mean]
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [block1SE,block2SE,block3SE,block4SE,block5SE,block6SE]
ax.errorbar(names,values,sevalues,color='black',ls='none')

#Plot distance differences for manuscript

plt.figure()
ax = plt.subplot()
TEST = pd.read_csv('/Users/macbook/Desktop/BETAVALUES1303.csv')
block1 = TEST[['18','Participant Number','Index']][TEST['Index']=='Sn(1) predpreynear']['18']
block2 = TEST[['18','Participant Number','Index']][TEST['Index']=='Sn(1) predpreymid']['18']
block3 = TEST[['18','Participant Number','Index']][TEST['Index']=='Sn(1) predpreyfar']['18']
block1mean = block1.mean()
block2mean = block2.mean()
block3mean = block3.mean()
block1SE = sem(block1)
block2SE = sem(block2)
block3SE = sem(block3)
names = ['Near','Mid','Far']
values = [block1mean, block2mean, block3mean]
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [block1SE,block2SE,block3SE]
ax.errorbar(names,values,sevalues,color='black',ls='none')

#Plot heading direction differences for manuscript

plt.figure()
ax = plt.subplot()
TEST = pd.read_csv('/Users/macbook/Desktop/BETAVALUES1403.csv')
block1 = TEST[['15','Participant Number','Index']][TEST['Index']=='Sn(1) preyahead']['15']
block2 = TEST[['15','Participant Number','Index']][TEST['Index']=='Sn(1) preyside']['15']
block3 = TEST[['15','Participant Number','Index']][TEST['Index']=='Sn(1) preybehind']['15']
block1mean = block1.mean()
block2mean = block2.mean()
block3mean = block3.mean()
block1SE = sem(block1)
block2SE = sem(block2)
block3SE = sem(block3)
names = ['Ahead','Side','Behind']
values = [block1mean, block2mean, block3mean]
ax.bar(names,values,color='grey',width=0.5)
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylabel('Beta')
sevalues = [block1SE,block2SE,block3SE]
ax.errorbar(names,values,sevalues,color='black',ls='none')